# RuleRAG_basic_v1.0.ipynb — TRPG 룰북 RAG (기본프로젝트 완성본)

| 구간 | 내용
|---|---
| ⚙️ 설정 | 모델·경로·검색 비중·재시도 횟수 (**바꿀 값은 여기만**)
| 0~2 | 설치, Ollama, 모델 받기, 드라이브
| 3~8 | 청크·임베딩·FAISS·질문 세트·번역·BM25·검색·캐시 (**검색 v1, 건드리지 않음**)
| 9~10 | 답변 도구 + **답변 파이프라인** (선택 → 검증 → 설명 → 검사)
| ▶ 질문해 보기 | `show_answer(질문)`
| 🧪 실험·평가 | 개발셋 점수, 선택·검증·충실성 실험, 용어표 켬/끔
| 🔒 최종 세트 | golden_final — 잠가 둔 셀
| 부록 | 옛 코드·기록 (전부 주석)

**흐름 한 줄**: 질문 → `retrieve`(후보 5개) → `select_chunk`(번호 하나) → `verify_chunk`(예/아니오) → `explain_chunk`(원문+설명) → `check_explain` + `check_faithful` → 화면

## ⚙️ 설정값

In [1]:
# Ex00. 설정값
# 모델
OLLAMA_URL = "http://localhost:11434"
EMBED_MODEL = "bge-m3"
GEN_MODEL = "qwen2.5:14b"
TRANSLATE_MODEL = GEN_MODEL

# 경로
DATA = "/content/drive/MyDrive/KT 기본프로젝트"
OUT_PATH = f"{DATA}/rules_new.jsonl"
EMB_PATH = f"{DATA}/rules_new_emb.npy"
CACHE_PATH = f"{DATA}/cache_dev_14b.json"

# 검색
TOP_K = 5
POOL_K = 10
SEARCH_WEIGHTS = [1, 1, 1, 0.5, 0.5]
RRF_C = 60
TRANSLATE_CTX = 4096

# 답변
GEN_SEED = 42
GEN_CTX = 8192
SELECT_CHARS = 300
VERIFY_CHARS = 800
CTX_CHARS = 2000
EXPLAIN_TRIES = 3
CHAT_ATTEMPTS = 3
DEV_VIEW = True


## 0. 설치

In [2]:
%%writefile /content/requirements_rb.txt
pymupdf==2.0
requests==2.32.4
faiss-cpu==1.15.1

Writing /content/requirements_rb.txt


In [3]:
# 설치
!pip install -r /content/requirements_rb.txt -q
!pip freeze | grep -E "pymupdf|requests|faiss"

faiss-cpu==1.15.1
pymupdf==2.0
pymupdf4llm==2.0
pymupdf_layout==2.0
requests==2.32.4
requests-oauthlib==2.0.0
requests-toolbelt==1.0.0


## 1. Ollama 설치 · 서버 · 모델 받기
모델은 **백그라운드로** 받아요. 다운로드가 도는 동안 다음 셀들을 진행해도 돼요.

In [4]:
!apt-get install -y -qq zstd
!curl -fsSL http://ollama.com/install.sh | sh
!ollama --version

Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [5]:
# Ex00-b. import · Ollama 서버 켜기
import os, re, json, time, math, subprocess
from collections import Counter
import numpy as np, requests, faiss
from tqdm.auto import tqdm
from google.colab import drive, files

def wait_server(url, seconds=120):
    for _ in range(seconds):
        try:
            if requests.get(url, timeout=2).status_code == 200:
                print("서버 준비 완료:", url)
                return
        except Exception:
            pass
        time.sleep(1)
    print("서버가 아직 준비되지 않았습니다. ollama.log를 확인하세요")

os.environ["OLLAMA_KEEP_ALIVE"] = "1h"
subprocess.Popen("nohup ollama serve > ollama.log 2>&1 &", shell=True)
wait_server(OLLAMA_URL)

서버 준비 완료: http://localhost:11434


In [6]:
# 모델 2개(임베딩 + 14B)를 백그라운드로 차례로 받기
subprocess.Popen(f'nohup sh -c "ollama pull {EMBED_MODEL} && ollama pull {GEN_MODEL}" > pull.log 2>&1 &', shell=True)
print("다운로드 시작. 진행 확인은 아래 셀")

다운로드 시작. 진행 확인은 아래 셀


In [7]:
# 모델이 다 받아질 때까지 기다리기 (최대 20분)
t0 = time.time()
while time.time() - t0 < 1200:
    out = subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout
    if GEN_MODEL in out and EMBED_MODEL in out:
        print(f"모델 준비 완료 ({int(time.time() - t0)}초 기다림)")
        break
    print("아직 받는 중...", int(time.time() - t0), "초")
    time.sleep(15)
else:
    print("20분이 지나도 안 받아졌어요. pull.log를 확인하세요:")
    print(open("pull.log").read()[-500:])

아직 받는 중... 0 초
아직 받는 중... 15 초
아직 받는 중... 30 초
아직 받는 중... 45 초
아직 받는 중... 60 초
아직 받는 중... 75 초
모델 준비 완료 (90초 기다림)


## 2. 구글 드라이브 · 경로

In [8]:
drive.mount("/content/drive")
print(OLLAMA_URL, "| LLM(번역·생성 공통):", GEN_MODEL, "| 임베딩:", EMBED_MODEL)


Mounted at /content/drive
http://localhost:11434 | LLM(번역·생성 공통): qwen2.5:14b | 임베딩: bge-m3


## 3. 청크 읽기 · 임베딩(캐시) · FAISS 인덱스

In [9]:
# Ex01. 저장한 청크 읽기
chunks = [json.loads(line) for line in open(OUT_PATH, encoding="utf-8")]
chunks = [c for c in chunks if c["meta"]["type"] == "rule"]
print("검색 대상 청크:", len(chunks))

# 검색 대상 청크: 1957

검색 대상 청크: 1957


In [10]:
# Ex02. 임베딩
def embed_batch(texts, model=EMBED_MODEL):
    res = requests.post(
        f"{OLLAMA_URL}/api/embed",
        json={"model": model, "input": texts, "options": {"num_ctx": 8192}},
        timeout=300,
    )
    res.raise_for_status()
    return res.json()["embeddings"]

BATCH = 32
if os.path.exists(EMB_PATH) and np.load(EMB_PATH).shape[0] == len(chunks):
    emb = np.load(EMB_PATH)
    print("캐시 사용:", emb.shape)
else:
    vecs = []
    for i in tqdm(range(0, len(chunks), BATCH)):
        vecs += embed_batch([c["text"] for c in chunks[i:i + BATCH]])
    emb = np.array(vecs, dtype="float32")
    np.save(EMB_PATH, emb)
    print("새로 계산:", emb.shape)

# 캐시 사용: (1957, 1024)

캐시 사용: (1957, 1024)


In [11]:
# Ex03. FAISS 인덱스 만들기
vecs = emb.copy()
faiss.normalize_L2(vecs)
index = faiss.IndexFlatIP(vecs.shape[1])
index.add(vecs)
print("인덱스에 들어간 벡터 수:", index.ntotal)

# 인덱스에 들어간 벡터 수: 1957

인덱스에 들어간 벡터 수: 1957


## 4. 평가 도구와 질문 세트
정답은 질문과 한 쌍(`q`, `want`)으로 파일에 두고, 검색 함수에는 `q`만 넣고 `want`는 채점에서만 써요.

- 개발셋: `golden`(영어 25) + `golden_new`(한국어 13) = **38문항** ← 설명 개선은 여기서만
- `golden_ko`(28)는 이미 오염된 옛 개발셋 (검색 점수 참고용)
- `golden_final`(60)은 최종 측정용 → 14번 구간에서만 사용

In [12]:
# Ex04. 정답 비교 도구 + 질문 세트 읽기
ALIAS = {"Grappl": {"grappled", "grappling"}}

def norm(title):
    title = re.sub(r"\s*\[[^\]]*\]$", "", title)      # 이름 끝의 [..] 제거
    title = re.sub(r"\s*\([^)]*\)$", "", title)       # 이름 끝의 (..) 제거
    return title.strip().lower()

def is_hit(title, want):
    return norm(title) == norm(want) or norm(title) in ALIAS.get(want, ())

def wants_of(g):
    return [g["want"]] if isinstance(g["want"], str) else g["want"]

def rank_in(found, g):
    titles = [c["meta"]["title"] for c in found]
    rank = next((i + 1 for i, t in enumerate(titles) if any(is_hit(t, w) for w in wants_of(g))), None)
    return rank, titles

def counts(ranks):
    return (sum(r == 1 for r in ranks),
            sum(r is not None and r <= 3 for r in ranks),
            sum(r is not None for r in ranks))

def evaluate(fn, items, show_miss=False):
    ranks = []
    for g in tqdm(items, leave=False):
        found = fn(g["q"])
        r, titles = rank_in(found, g)
        ranks.append(r)
        if show_miss and r != 1:
            print(f"[{r}] {g['q']}\n     정답: {g['want']} | 검색: {titles[:3]}")
    return counts(ranks)

def load_set(name):
    path = f"{DATA}/{name}.json"
    return json.load(open(path, encoding="utf-8")) if os.path.exists(path) else []

golden       = load_set("golden")          # 영어 25문항
golden_ko    = load_set("golden_ko")       # 한국어 28문항
golden_new   = load_set("golden_new")      # 한국어 13문항
golden_final = load_set("golden_final")    # 최종 60문항

# 정답 제목이 룰북에 실제로 있는지 확인
titles_all = {norm(c["meta"]["title"]) for c in chunks}
for name, items in {"golden": golden, "golden_ko": golden_ko, "golden_new": golden_new, "golden_final": golden_final}.items():
    miss = [w for g in items for w in wants_of(g) if norm(w) not in titles_all and w not in ALIAS]
    print(f"{name}: {len(items)}문항 | 룰북에 없는 정답: {miss}")
sets3 = {"영어": golden, "한국어ko": golden_ko, "한국어new": golden_new}

# golden: 25문항 | 룰북에 없는 정답: []
# golden_ko: 28문항 | 룰북에 없는 정답: []
# golden_new: 13문항 | 룰북에 없는 정답: []
# golden_final: 60문항 | 룰북에 없는 정답: []

golden: 25문항 | 룰북에 없는 정답: []
golden_ko: 28문항 | 룰북에 없는 정답: []
golden_new: 13문항 | 룰북에 없는 정답: []
golden_final: 60문항 | 룰북에 없는 정답: []


In [14]:
# Ex04-b. 선택 평가용 느슨한 제목 비교 (단수/복수만 같은 것)
def stem(title):
    return re.sub(r"s\b", "", norm(title))

def is_hit_loose(title, want):
    return stem(title) == stem(want) or norm(title) in ALIAS.get(want, ())

## 5. 번역 (한국어 질문 → 영어 검색어)
- 번역①: 힌트 없는 일반 번역(예시 3개)
- 번역②: 규칙 용어 27개 선택지 번역
- 호출은 **14B, temperature 0, seed 42, num_ctx 4096**. (옛 v1은 7B 번역이었고 num_ctx를 바꾸면 번역도 달라져요.)

In [15]:
# Ex05-a. 번역 호출
def ask_translate(prompt, model=None, temperature=0.0):
    res = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={"model": model or TRANSLATE_MODEL,
              "messages": [{"role": "user", "content": prompt}],
              "stream": False,
              "options": {"temperature": temperature, "seed": 42, "num_ctx": TRANSLATE_CTX}},
        timeout=300,
    )
    res.raise_for_status()
    return res.json()["message"]["content"]

In [16]:
# Ex05-b. 번역 프롬프트 두 가지
def clean_query(s):                                 # 첫 줄만, "Query:" 접두어와 따옴표 제거
    s = s.strip().split("\n")[0]
    s = re.sub(r"^\s*query\s*:\s*", "", s, flags=re.I)
    return s.strip().strip("\"'")

# 번역1 : 일반 번역
REWRITE_TEMPLATE = """Convert the user's question about D&D 5.2 rules into a short English search query.
Use the official rule terms that the rulebook would use.
Output only the query, nothing else.

Question: 숙련 보너스는 레벨마다 얼마야?
Query: Proficiency Bonus by character level

Question: 레벨업하려면 경험치가 얼마나 필요해?
Query: Experience Points needed to advance level

Question: 방어도는 어떻게 계산해?
Query: Armor Class calculation

Question: {question}
Query:"""

def translate1(q):
    return clean_query(ask_translate(REWRITE_TEMPLATE.format(question=q)))

# 상태이상15개 + 행동12개 (전부) / 룰북
RULE_TERMS = ["Blinded", "Charmed", "Deafened", "Exhaustion", "Frightened", "Grappled", "Incapacitated",
              "Invisible", "Paralyzed", "Petrified", "Poisoned", "Prone", "Restrained", "Stunned", "Unconscious",
              "Attack action", "Dash action", "Disengage action", "Dodge action", "Help action", "Hide action",
              "Influence action", "Magic action", "Ready action", "Search action", "Study action", "Utilize action"]

REWRITE2 = """Translate the user's D&D question into a short English search query for the D&D rules.
If the question is about one of these official rule terms, use that term exactly as written.
Official terms: {terms}
Output only the query, nothing else.

Question: {q}
Query:"""

def translate2(q):
    return clean_query(ask_translate(REWRITE2.format(terms=", ".join(RULE_TERMS), q=q)))

print(translate1("겁에 질리면 뭘 못 해?"), "|", translate2("겁에 질리면 뭘 못 해?"))

# Effects of being Frightened | frightened action restrictions

Effects of being Frightened | frightened action restrictions


## 6. 키워드 검색 BM25 (직접 구현, 새 설치 없음)
임베딩은 **뜻**, BM25는 **글자 그대로의 단어**를 비교해서 서로 보완돼요. 이름이 정확한 질문(`Two-Weapon Fighting`)에 강해요. 제목을 두 번 넣어 제목 단어에 가중치를 줘요.

In [17]:
# Ex06. BM25 직접 만들기
STOP = set("a an the of to in on and or is are be can you your it its as at by for with that this if when how what do does".split())

def tokenize(s):
    return [w for w in re.findall(r"[a-z0-9]+", s.lower()) if w not in STOP]

class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.tf = [Counter(d) for d in docs]
        self.len = np.array([len(d) for d in docs], dtype="float32")
        self.avg = self.len.mean()
        df = Counter(w for d in docs for w in set(d))
        N = len(docs)
        self.idf = {w: math.log(1 + (N - n + 0.5) / (n + 0.5)) for w, n in df.items()}
        # tf(단어많이나옴), idf(희귀한단어), len(짧은청크) -> 점수 높음

    def scores(self, query):
        s = np.zeros(len(self.tf), dtype="float32")
        for w in set(tokenize(query)):
            if w not in self.idf:
                continue
            for i, tf in enumerate(self.tf):
                f = tf.get(w, 0)
                if f:
                    s[i] += self.idf[w] * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * self.len[i] / self.avg))
                    # 희귀도 x 빈도효과 / 길이보정
        return s

bm25 = BM25([tokenize(c["meta"]["title"] + " " + c["meta"]["title"] + " " + c["text"]) for c in chunks])

def bm25_search(q, k=10):
    return [chunks[i] for i in np.argsort(-bm25.scores(q))[:k]]

print([c["meta"]["title"] for c in bm25_search("Two-Weapon Fighting", 3)])

# ['Two-Weapon Fighting', 'Great Weapon Fighting', 'Level 1: Fighting Style']

['Two-Weapon Fighting', 'Great Weapon Fighting', 'Level 1: Fighting Style']


## 7. 최종 검색 함수 `retrieve` (임베딩 3갈래 + 키워드 2갈래 → 가중 RRF)
RRF: 각 검색의 순위를 `비중/(60+순위)` 점수로 바꿔 더해요. 임베딩 3표(원문 질문, 번역①, 번역②)는 비중 1.0, 키워드 2표(번역①, ②)는 비중 0.5예요.

In [18]:
# Ex07. 검색: 질문 하나 → 청크 5개
def dense_search(texts, k=5):
    v = np.array(embed_batch(texts), dtype="float32")
    faiss.normalize_L2(v)
    sc, ids = index.search(v, k)
    best = {}
    for srow, irow in zip(sc, ids):
        for s, i in zip(srow, irow):
            best[i] = max(best.get(i, -1.0), float(s))
    return [chunks[i] for i in sorted(best, key=best.get, reverse=True)[:k]]

def rrf_merge(lists, weights, k=TOP_K, c0=RRF_C):
    score, byid = {}, {}
    for lst, w in zip(lists, weights):
        for r, c in enumerate(lst, 1):
            score[c["id"]] = score.get(c["id"], 0) + w / (c0 + r)
            byid[c["id"]] = c
    return [byid[i] for i in sorted(score, key=score.get, reverse=True)[:k]]

def retrieve(q, k=TOP_K):
    en1, en2 = translate1(q), translate2(q)
    lists = [dense_search([t], POOL_K) for t in (q, en1, en2)] + [bm25_search(en1, POOL_K), bm25_search(en2, POOL_K)]
    return rrf_merge(lists, SEARCH_WEIGHTS, k), [q, en1, en2]


In [ ]:
# 검색 개선 사항
# 비중 조정 → [1,1,1,0.5,0.5]를 다른 조합으로 변경 (BM25 0.3/0.7, 번역2 가중 상향 등)
# RRF 상수 → c0=60을 20/100으로 변경
# 후보 폭 → 목록당 10개를 20개로 확대
# 번역2 프롬프트 → "확실하지 않으면 용어 넣지 말기" + 핵심 명사 유지 + 예시 1~2개
# 번역 횟수 → 번역③(키워드만 추출) 추가
# BM25 제목 가중 → 제목 2번을 1번·3번으로 변경
# 불용어 → 목록 확대 또는 제거
# 청크 분할 → 긴 청크를 나누거나 짧은 청크를 합침
# 임베딩 입력 → 청크에 제목·경로를 붙여서 임베딩
# 리랭커 → 재도입 (이미 하락, 기각 상태)

## 8. 번역·검색 캐시 (개발셋을 빠르게 + 런타임이 초기화돼도 유지)

**왜 필요해요?** T4에서는 `14B ↔ bge-m3`를 번갈아 부르면 모델이 메모리에서 교체돼서 질문당 18~20초가 걸려요. 번역과 검색 결과를 드라이브에 저장해 두면 개발셋은 질문당 약 6초로 줄어요.

**순서가 중요해요 (캐시 파일 이름 `cache_dev_14b.json`)**
1. 아래 첫 셀: 캐시 장치 설치 + 파일이 있으면 불러오기
2. 파일이 없거나 새 질문이 생기면 → 둘째 셀 `build_cache(...)` (번역만 먼저 → 검색만 나중에)
3. 저장은 둘째 셀 끝에서 해요. 개발셋 검색 점수(Ex08-c)는 🧪 실험 구간으로 옮겼어요

옛 캐시(`cache_dev.json`, `cache_dev_v1.json`)는 7B 번역 기준이라 쓰지 않아요.

In [19]:
# Ex08-a. 캐시 장치 : 번역, 검색 결과를 기억
by_id = {c["id"]: c for c in chunks}
_tr_cache, _sf_cache = {}, {}

if "_orig_tr1" not in globals():
    _orig_tr1, _orig_tr2, _orig_retrieve = translate1, translate2, retrieve

def translate1(q):
    if ("1", q) not in _tr_cache: _tr_cache[("1", q)] = _orig_tr1(q)
    return _tr_cache[("1", q)]
def translate2(q):
    if ("2", q) not in _tr_cache: _tr_cache[("2", q)] = _orig_tr2(q)
    return _tr_cache[("2", q)]
def retrieve(q, k=TOP_K):
    if (q, k) not in _sf_cache: _sf_cache[(q, k)] = _orig_retrieve(q, k)
    return _sf_cache[(q, k)]

def save_cache():
    json.dump({
        "tr": {f"{k[0]}|{k[1]}": v for k, v in _tr_cache.items()},
        "sf": {f"{k[0]}|{k[1]}": {"ids": [c["id"] for c in v[0]], "texts": v[1]}
               for k, v in _sf_cache.items()},
    }, open(CACHE_PATH, "w", encoding="utf-8"), ensure_ascii=False)
    print("저장:", len(_tr_cache), "번역 /", len(_sf_cache), "검색")

def load_cache():
    if not os.path.exists(CACHE_PATH):
        print("캐시 파일 없음 → 아래 build_cache(dev)로 만들어요")
        return
    d = json.load(open(CACHE_PATH, encoding="utf-8"))
    for k, v in d["tr"].items():
        _tr_cache[tuple(k.split("|", 1))] = v
    for k, v in d["sf"].items():
        q, kk = k.rsplit("|", 1)
        _sf_cache[(q, int(kk))] = ([by_id[i] for i in v["ids"]], v["texts"])
    print("불러옴:", len(_tr_cache), "번역 /", len(_sf_cache), "검색")

def unload(model):
    requests.post(f"{OLLAMA_URL}/api/generate", json={"model": model, "keep_alive": 0})

dev = golden + golden_new      # 개발셋 38문항
load_cache()

# 불러옴: 132 번역 / 66 검색

불러옴: 132 번역 / 66 검색


In [20]:
# Ex08-b. 캐시 만들기: 1.번역 → 2.검색 / 이미 캐시된 질문은 건너뜀
def build_cache(items):
    qs = [g["q"] for g in items]
    todo_tr = [q for q in qs if ("1", q) not in _tr_cache or ("2", q) not in _tr_cache]
    if todo_tr:
        _orig_tr1(todo_tr[0])
        for q in tqdm(todo_tr, desc="번역(14B)"):
            translate1(q); translate2(q)
        unload(TRANSLATE_MODEL)
    todo_sf = [q for q in qs if (q, TOP_K) not in _sf_cache]
    for q in tqdm(todo_sf, desc="검색(bge-m3)"):
        retrieve(q)
    if todo_sf:
        unload(EMBED_MODEL)
    print(f"완료. 번역 새로 {len(todo_tr)}개, 검색 새로 {len(todo_sf)}개 → 캐시 번역 {len(_tr_cache)} / 검색 {len(_sf_cache)}")

build_cache(dev + golden_ko)
save_cache()

# 검색(bge-m3): 0/0 [00:00<?, ?it/s]
# 완료. 번역 새로 0개, 검색 새로 0개 → 캐시 번역 132 / 검색 66
# 저장: 132 번역 / 66 검색

검색(bge-m3): 0it [00:00, ?it/s]

완료. 번역 새로 0개, 검색 새로 0개 → 캐시 번역 132 / 검색 66
저장: 132 번역 / 66 검색


## 9. 답변 생성 도구 (14B)
근거에 번호를 붙여 **번호를 먼저 고르고** 그 근거에서만 인용하게 해요. 용어표(`TERM_MAP`)는 검색용이 아니라 **답변의 한국어 표기를 통일**하는 용도예요.

In [21]:
# Ex09-a. 용어표 (한국어 = 룰북 영어 용어) — 답변용
TERM_V1 = {
    # 판정·전투
    "이점": "Advantage", "불리점": "Disadvantage", "능력 판정": "Ability Check",
    "공격 굴림": "Attack Roll", "내성 굴림": "Saving Throw", "기회 공격": "Opportunity Attack",
    "반응": "Reaction", "추가 행동": "Bonus Action", "집중": "Concentration",
    "이니셔티브": "Initiative", "저항": "Resistance", "취약": "Vulnerability",
    "엄폐": "Cover", "치명타": "Critical Hit", "숙련": "Proficiency", "전문화": "Expertise",
    "긴 휴식": "Long Rest", "짧은 휴식": "Short Rest",
    # 능력치
    "근력": "Strength", "민첩": "Dexterity", "건강": "Constitution",
    "지능": "Intelligence", "지혜": "Wisdom", "카리스마": "Charisma",
    # 피해 종류 13
    "산성": "Acid", "타격": "Bludgeoning", "냉기": "Cold", "화염": "Fire", "역장": "Force", "번개": "Lightning",
    "사령": "Necrotic", "관통": "Piercing", "독": "Poison", "정신": "Psychic", "광휘": "Radiant", "참격": "Slashing", "천둥": "Thunder",
    # 클래스 12
    "바바리안": "Barbarian", "바드": "Bard", "클레릭": "Cleric", "드루이드": "Druid", "파이터": "Fighter", "몽크": "Monk",
    "팔라딘": "Paladin", "레인저": "Ranger", "로그": "Rogue", "소서러": "Sorcerer", "워록": "Warlock", "위저드": "Wizard",
    # 주문 학파 8 + 주문 정보 칸
    "방호": "Abjuration", "소환": "Conjuration", "예지": "Divination", "환혹": "Enchantment", "방출": "Evocation",
    "환영": "Illusion", "강령": "Necromancy", "변환": "Transmutation",
    "시전 시간": "Casting Time", "사거리": "Range", "구성요소": "Components", "지속 시간": "Duration",
    "음성": "Verbal", "동작": "Somatic", "물질": "Material",
    # 룰북 용어집 -> 의미있나?
    "능력치와 수정치": "Ability Score and Modifier", "행동": "Action", "모험": "Adventure", "성향": "Alignment", "아군": "Ally",
    "효과 범위": "Area of Effect", "방어도": "Armor Class", "방어구 숙련": "Armor Training", "태도": "Attitude", "조율": "Attunement",
    "맹시": "Blindsight", "피투성이": "Bloodied", "밝은 빛": "Bright Light", "굴착 이동 속도": "Burrow Speed", "캠페인": "Campaign",
    "캔트립": "Cantrip", "운반 용량": "Carrying Capacity", "도전 지수": "Challenge Rating", "등반 속도": "Climb Speed",
    "상태": "Condition", "생물": "Creature", "생물 유형": "Creature Type", "원뿔": "Cone", "정육면체": "Cube", "원기둥": "Cylinder",
    "저주": "Curses", "D20 판정": "D20 Test", "피해": "Damage", "피해 굴림": "Damage Roll", "피해 임계값": "Damage Threshold",
    "피해 유형": "Damage Types", "어둠": "Darkness", "암시야": "Darkvision", "질주": "Dash", "사망": "Dead",
    "죽음 내성 굴림": "Death Saving Throw", "탈수": "Dehydration", "험지": "Difficult Terrain", "난이도 등급": "Difficulty Class",
    "희미한 빛": "Dim Light", "이탈": "Disengage", "회피": "Dodge", "발산": "Emanation", "조우": "Encounter", "적": "Enemy",
    "경험치": "Experience Points", "낙하": "Falling", "비행 속도": "Fly Speed", "비행": "Flying", "우호적": "Friendly",
    "위험 요소": "Hazard", "치유": "Healing", "심하게 가려짐": "Heavily Obscured", "도움": "Help", "영웅적 영감": "Heroic Inspiration",
    "은신": "Hide", "높이뛰기": "High Jump", "체력 주사위": "Hit Point Dice", "체력": "Hit Points", "적대적": "Hostile",
    "부양": "Hover", "면역": "Immunity", "임시변통 무기": "Improvised Weapons", "무관심": "Indifferent", "영향": "Influence",
    "도약": "Jumping", "약간 가려짐": "Lightly Obscured", "직선": "Line", "멀리뛰기": "Long Jump", "마법": "Magic",
    "마법 효과": "Magical Effect", "영양실조": "Malnutrition", "몬스터": "Monster", "비플레이어 캐릭터": "Nonplayer Character",
    "물체": "Object", "점유 공간": "Occupied Space", "수동 감지": "Passive Perception", "하루당": "Per Day",
    "플레이어 캐릭터": "Player Character", "소유": "Possession", "도달 거리": "Reach", "대기": "Ready", "의식": "Ritual",
    "내림": "Round Down", "내성": "Save", "수색": "Search", "변신": "Shape-Shifting", "동시 효과": "Simultaneous Effects",
    "크기": "Size", "기술": "Skill", "속도": "Speed", "주문": "Spell", "주문 공격": "Spell Attack", "주문 시전 매개체": "Spellcasting Focus",
    "구": "Sphere", "안정": "Stable", "스탯 블록": "Stat Block", "연구": "Study", "질식": "Suffocation", "수영 속도": "Swim Speed",
    "수영": "Swimming", "대상": "Target", "텔레파시": "Telepathy", "순간이동": "Teleportation", "임시 체력": "Temporary Hit Points",
    "진동감지": "Tremorsense", "진시": "Truesight", "맨손 타격": "Unarmed Strike", "빈 공간": "Unoccupied Space", "무기": "Weapon",
    "무기 공격": "Weapon Attack",
}

TERM_MAP = dict(TERM_V1)
print("용어표 v1:", len(TERM_MAP))

TERMS = ", ".join(f"{en}={ko}" for ko, en in TERM_MAP.items())
TERM_REV = {en.lower(): ko for ko, en in TERM_MAP.items()}

def terms_for(text):
    out = []
    for ko, en in TERM_MAP.items():
        pat = rf"\b{re.escape(en)}s?\b" if " " in en else rf"\b{re.escape(en)}\b"
        if re.search(pat, text, re.I if " " in en else 0):
            out.append(f"{en}={ko}")
    return ", ".join(out) or "(없음)"

# 전체 제목에 대한 용어표 사용 검토 -> 의미가 있나? --> 코드 내 삽입이 아닌 파일로 저장

용어표 v1: 175


In [24]:
# Ex09-b. 호출 함수 + 깨짐/혼합/분수 검사
SYSTEM = ("You are a D&D rules assistant. Write every answer in Korean (Hangul). "
          "Never use Chinese characters or Chinese words. "
          "Also never use Japanese, Vietnamese, Russian, Thai, or any other language or script. "
          "Exceptions: English D&D terms in parentheses only when the instructions ask for them, "
          "and the English source quote when asked to copy it.")

def _chat(prompt, model=None, system=SYSTEM, temperature=0.0, seed=GEN_SEED, max_tokens=None):
    opts = {"num_ctx": GEN_CTX, "temperature": temperature}
    if seed is not None: opts["seed"] = seed
    if max_tokens: opts["num_predict"] = max_tokens
    res = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={"model": model or GEN_MODEL,
              "messages": [{"role": "system", "content": system},
                           {"role": "user", "content": prompt}],
              "stream": False, "options": opts},
        timeout=600,
    )
    res.raise_for_status()
    return res.json()["message"]["content"]

def ask_explain(prompt, model=None, temperature=0.1, seed=GEN_SEED):
    return _chat(prompt, model, SYSTEM, temperature, seed)

def ask_short(prompt, model=None, system=SYSTEM):
    return _chat(prompt, model, system, 0.0, GEN_SEED, max_tokens=20)


BROKEN = re.compile(r"[\u4e00-\u9fff\u3040-\u30ff\u1ea0-\u1ef9\u0300-\u036făâêôơưđĐ\ufffd]")
# re.compile + 유니코드 범위 = 이 글자들 중 하나라도 있으면 깨짐
#  \u4e00-\u9fff 한자 / \u3040-\u30ff 일본어 / \u1ea0-\u1ef9 및 ăâêôơưđ 베트남어 / \ufffd 깨진 글자
def is_broken(text):
    return bool(BROKEN.search(text))

LEAK = re.compile(r"(^|\n)\s*(user|assistant|system)\s*(\n|:)", re.I)
# 줄 처음에 "user" 같은 말이 새어 나왔는지

MIXED = re.compile(r"[가-힣][A-Za-z]")         # 한글 바로 뒤에 영어 철자가 붙음 (예: "이점Advantage", "스티일")
FRACTION = re.compile(r"\d+분의\s*\d+")        # "3분의 2" — 14B가 Three-Quarters를 뒤집는 오류가 있었어요

def bad_output(text):
    return is_broken(text) or bool(MIXED.search(text)) or bool(FRACTION.search(text))

# 서버 오류 → try/except + 재시도 1~2번 (Ollama가 끊겨도 챗봇이 안 멈춰요, 배포 전 필수)
# 컨텍스트 → GEN_CTX 8192를 4096으로 변경 (빨라지지만 출력이 달라질 수 있어 개발셋 확인 필요)
# 스트리밍 → stream: True 변경 (서버 붙일 때, 나중)


In [23]:
# Ex09-c. 검사 도구
def expl_part(text):
    m = re.search(r"설명\s*[:：]\s*(.*)", text, flags=re.S) # 답변에서 "설명:" 뒤에 오는 글을 찾기
    return m.group(1).strip() if m else text

NUM = re.compile(r"\d+(?:d\d+)?", re.I)            # 숫자 또는 주사위 표기(2d6)
def numbers(text):
    return [n.lower() for n in NUM.findall(text)]

# 영어 숫자 단어 → 숫자
W2N = {"one": "1", "two": "2", "three": "3", "four": "4", "five": "5", "six": "6", "seven": "7",
       "eight": "8", "nine": "9", "ten": "10", "half": "1 2", "quarter": "1 4", "quarters": "4"}
W2N.update({"twice": "2", "double": "2", "doubled": "2"})

norm_txt = lambda s: re.sub(r"\s+", " ", re.sub(r"[\"'“”‘’]", "", s)).strip().lower()
# 따옴표·공백 차이 무시하고 비교


## 10. 답변 파이프라인 (선택 → 검증/거절 → 설명 → 자동 검사)

```
질문 → retrieve(5개) → select_chunk(번호) → verify_chunk(예/아니오)
        └ 아니오 → 거절 ─┐
        └ 예 → explain_chunk(원문+설명) → check_explain(코드 검사) → check_faithful(근거에 없는 내용?)
                                      └ 걸리면: 개발 모드 = 숨기고 근거 본문 표시 / 챗봇 모드 = 다른 seed로 재시도 → 그래도 걸리면 "확실하지 않아요"
```
- 원문·제목·페이지는 **모델이 아니라 코드가** 붙여요 (원문이 근거에 실제로 있는지 코드가 확인).
- 설명 단계는 선택된 **근거 하나만** 봐요. 용어표도 그 근거에 나온 용어만 넣어요 (`terms_for`).
- 정책 A: 검증이 "아니오"면 거절해요.


In [25]:
# Ex10-a. 근거 선택: 후보 5개 중 번호 하나 고르기
SELECT_TEMPLATE = """다음은 D&D 규칙 질문과 후보 근거입니다. 질문에 직접 답하는 근거의 번호 하나만 고르세요.
- 질문이 묻는 규칙 자체를 설명하는 근거를 고르세요. 이름만 비슷하거나 예시로 지나가듯 언급되는 근거는 고르지 마세요.
- 숫자 하나만 출력하세요.

### 질문
{question}

### 후보 근거
{cands}

### 번호"""

def select_chunk(q, found, model=None):
    cands = "\n".join(f"[{n}] {c['meta']['title']}: {c['text'][:SELECT_CHARS].replace(chr(10), ' ')}"
                      for n, c in enumerate(found, 1))
    out = ask_short(SELECT_TEMPLATE.format(question=q, cands=cands), model)
    m = re.search(r"\d+", out)
    n = int(m.group()) if m else 1
    return n if 1 <= n <= len(found) else 1


In [28]:
# Ex10-b. 설명하기 : 챗봇이 답을 쓰고, 코드가 그 답을 확인
EXPLAIN_TEMPLATE = """당신은 D&D 규칙 안내자입니다. 아래 [근거] 하나만 사용해 질문에 답하세요.
1. 근거에 적힌 내용만 사용하세요. 숫자, 조건, 이점/불리점을 빼거나 바꾸지 마세요.
2. 용어는 이 표의 한국어를 쓰고 영어 원문을 괄호로 함께 적어. 같은 용어는 항상 같은 한국어로 써: {terms}
   '이점'은 Advantage에만 쓰고, bonus는 '보너스', roll은 '굴림', check는 '판정'으로 써.
3. 답변은 "원문:", "설명:" 두 줄로 쓰세요. 원문은 질문에 직접 답하는 문장 1~3개를 영어 그대로, 끝까지, 한 줄로 복사하세요.
   설명은 근거에 있는 내용만 한국어로 쓰세요. 질문이 묻는 내용은 빠짐없이 쓰고(목록이면 항목 전부, 이점/불리점은 누구의 굴림인지 구분), 묻지 않은 다른 규칙이나 예시는 쓰지 마세요.
   인사, 서론, 되묻기는 쓰지 마세요. "원문:"과 "설명:"은 각각 한 번만 쓰세요.

### 근거
{context}

### 질문
{question}

### 답변"""
RULE_EN = "용어는 이 표의 한국어를 쓰고 영어 원문을 괄호로 함께 적어. 같은 용어는 항상 같은 한국어로 써: {terms}"
RULE_KO = "용어는 이 표의 한국어만 쓰세요. 영어는 괄호로도 적지 마세요. 같은 용어는 항상 같은 한국어로 써: {terms}"
assert RULE_EN in EXPLAIN_TEMPLATE, "2번 규칙 문구가 달라요"

def explain_chunk(question, k=TOP_K, tries=EXPLAIN_TRIES, temp0=0.1, seed0=40, pre=None, en_terms=True):    # seed0/pre: 챗봇 재시도용 (기본값이면 이전과 같아요)
    found, texts = pre[:2] if pre else retrieve(question, k)
    n = pre[2] if pre else select_chunk(question, found)          # 1. 번호 고르기
    c = found[n - 1]
    ctx = f"({c['meta']['title']} p.{c['meta']['page']}) {c['text'][:CTX_CHARS]}"
    tmpl = EXPLAIN_TEMPLATE if en_terms else EXPLAIN_TEMPLATE.replace(RULE_EN, RULE_KO)
    prompt = tmpl.format(terms=terms_for(c["text"]), context=ctx, question=question)
    for t in range(1, tries + 1):                                 # 2. 설명
        out = ask_explain(prompt, temperature=temp0 + 0.3 * (t - 1), seed=seed0 + t)
        if not bad_output(out):
            break
    m = re.search(r"원문\s*[:：]\s*(.+)", out)
    orig = m.group(1).strip() if m else ""
    verified = bool(orig) and norm_txt(orig) in norm_txt(c["text"])
    return {"title": c["meta"]["title"], "page": c["meta"]["page"], "text": c["text"], "orig_ok": verified,
            "orig": orig if verified else c["text"].split("\n", 1)[-1][:300],
            "path": c["text"].split("\n", 1)[0],
            "explain": expl_part(out), "picked": n, "queries": texts[1:], "tries": t}

def check_explain(r):
    ok = r["orig_ok"]
    complete = ok and r["orig"].rstrip().endswith((".", "!", "?", '"'))
    src = set(numbers(r["text"])) | {d for w in re.findall(r"[a-z]+", r["text"].lower())
                                      if w in W2N for d in W2N[w].split()}
    bad_nums = [x for x in numbers(r["explain"]) if x not in src]
    term_miss = [ko for ko, en in TERM_MAP.items()
                 if ok and re.search(rf"\b{re.escape(en)}\b", r["orig"]) and ko not in r["explain"]]
    # 근거에 Advantage가 나온 횟수보다 '이점'이 더 많이 나오면 bonus 같은 걸 이점으로 쓴 것
    return {"원문_검증": ok, "원문_완결": complete, "근거에_없는_숫자": bad_nums, "용어_누락": term_miss,
            "혼합_철자": bool(MIXED.search(r["explain"])), "분수": FRACTION.findall(r["explain"]),
            "형식_누출": bool(re.search(r"(원문|설명)\s*[:：]", r["explain"]))}

In [29]:
# Ex10-c. 근거 검증 · 충실성 검사 · 최종 함수 `answer` / `show_answer`
VERIFY_TEMPLATE = """다음 D&D 규칙 질문과 근거입니다. 이 근거가 질문에 직접 답하는 규칙 내용을 담고 있으면 '예', 이름만 비슷하거나 다른 주제이면 '아니오'라고만 답하세요.
질문은 한국어, 근거는 영어예요. 표현이 달라도 같은 규칙을 다루면 '예'예요.

### 질문
{question}
(영어 검색어: {en})

### 근거
{title}: {text}

### 답 (예/아니오)"""

def verify_chunk(q, c, en=""):
    out = ask_short(VERIFY_TEMPLATE.format(question=q, en=en or "-", title=c["meta"]["title"],
                                           text=c["text"][:VERIFY_CHARS]))
    return re.sub(r"[\s\*'\"]", "", out).startswith(("예", "네"))

FAITH_TEMPLATE = """다음은 D&D 규칙 근거(영어)와 그것을 한국어로 설명한 글입니다.
설명에 근거에 없는 내용(지어낸 사실, 근거와 다른 숫자·조건)이 있으면 '있음', 없으면 '없음'이라고만 답하세요.

### 근거
{src}

### 설명
{exp}

### 답 (있음/없음)"""

def check_faithful(r):
    out = ask_short(FAITH_TEMPLATE.format(src=r["text"][:CTX_CHARS], exp=r["explain"]))
    return out.strip().startswith("없음")

def collect_warnings(r, chk, check_faith=True, show_orig=True):
    w = []
    if not chk["원문_검증"]: w.append("원문 검증 실패")
    elif show_orig and not chk["원문_완결"]: w.append("원문이 문장 중간에서 끊김")
    if chk["근거에_없는_숫자"]: w.append(f"근거에 없는 숫자 {chk['근거에_없는_숫자']}")
    if chk["분수"]: w.append(f"분수 표현 {chk['분수']}")
    if chk["형식_누출"]: w.append("형식 누출")
    if chk["혼합_철자"]: w.append("혼합 철자")
    if is_broken(r["explain"]): w.append("깨진 글자")
    if not w and check_faith and not check_faithful(r): w.append("근거에 없는 내용")
    return w

def sure_match(texts, c):
    t = norm(c["meta"]["title"])
    return any(re.search(rf"\b{re.escape(t)}\b", x.lower()) for x in texts[1:])

def passes_verify(q, c, texts):
    return sure_match(texts, c) or verify_chunk(q, c, texts[1])

def answer(question, attempts=1, show_orig=True):
    found, texts = retrieve(question)
    n = select_chunk(question, found)
    c = found[n - 1]
    if not passes_verify(question, c, texts):
        return {"status": "refuse", "candidates": [(x["meta"]["title"], x["meta"]["page"]) for x in found[:3]],
                "first_flagged": False, "attempts_used": 0, "warn": []}
    first_flagged, r, warn = None, None, []
    for a in range(attempts):
        r = explain_chunk(question, temp0=0.0 if a == 0 else 0.3, seed0=40 + 10 * a,
                pre=(found, texts, n), en_terms=show_orig)
        warn = collect_warnings(r, check_explain(r), show_orig=show_orig)
        if first_flagged is None: first_flagged = bool(warn)
        if not warn: break
    return {"status": "hidden" if warn else "ok", "warn": warn, "first_flagged": first_flagged, "attempts_used": a + 1,
            "title": r["title"], "page": r["page"], "path": r["path"], "orig": r["orig"],
            "body": r["text"].split("\n", 1)[-1][:600], "explain": r["explain"]}

def show_answer(question, dev_view=None):
    dev_view = DEV_VIEW if dev_view is None else dev_view
    r = answer(question, attempts=1 if dev_view else CHAT_ATTEMPTS, show_orig=dev_view)
    if r["status"] == "refuse":
        print("🙅 이 질문에 맞는 규칙 근거를 확실히 찾지 못했어요." + (f" 후보: {r['candidates']}" if dev_view else " 질문을 바꿔 보시겠어요?"))
    elif r["status"] == "hidden":
        if dev_view:
            print(f"⚠️ 설명을 숨겼어요 {r['warn']}\n출처: {r['title']} p.{r['page']} ({r['path']})\n[근거 본문] {r['body']}")
        else:
            print("확실하지 않아요. 질문을 조금 바꿔서 다시 물어봐 주세요.")
    else:
        if dev_view:
            print(f"출처: {r['title']} p.{r['page']} ({r['path']})\n원문: {r['orig']}\n설명: {r['explain']}")
        else:
            print(r["explain"])
    return r

### 질문 해 보기 (여기만 바꿔서 다시 실행)
`DEV_VIEW = True`면 원문·페이지·근거 본문까지, `False`면 챗봇처럼 설명만 보여 줘요. 캐시에 없는 새 질문은 모델 교체 때문에 첫 질문이 20초쯤 걸릴 수 있어요.

In [ ]:
question = "넘어졌을 때 일어나려면 이동력이 얼마나 들어?"
_ = show_answer(question)

# -----------------------------------------------------------------------

## 🧪 실험·평가
디버깅 회귀 : Ex16-d


In [ ]:
# Ex08-c. 번역을 14B로 바꾼 뒤의 개발셋 검색 점수 (옛 v1 = 7B 번역 기준과 비교만 해요)
V1_DEV = {"영어": (20, 24, 24), "한국어ko": (16, 23, 24), "한국어new": (5, 7, 9)}     # 7B 번역일 때 점수
tot = [0, 0, 0]
for name, items in sets3.items():
    got = counts([rank_in(retrieve(g["q"])[0], g)[0] for g in items])
    tot = [x + y for x, y in zip(tot, got)]
    print(f"{name:8s} 14B 번역 {got} | 7B 번역(v1) {V1_DEV[name]}")
print("합계(66):", tuple(tot), "| 7B 번역(v1): (41, 54, 57)")
save_cache()

# 영어       14B 번역 (21, 24, 25) | 7B 번역(v1) (20, 24, 24)
# 한국어ko    14B 번역 (15, 18, 23) | 7B 번역(v1) (16, 23, 24)
# 한국어new   14B 번역 (7, 10, 11) | 7B 번역(v1) (5, 7, 9)
# 합계(66): (43, 52, 59) | 7B 번역(v1): (41, 54, 57)
# 저장: 132 번역 / 66 검색

In [ ]:
# Ex12. 개발 3세트 검색 점수 (1등 / 3등 안 / 5등 안)
fn = lambda q: retrieve(q)[0]
tot = [0, 0, 0]
for name, items in sets3.items():
    c = evaluate(fn, items)
    tot = [a + b for a, b in zip(tot, c)]
    print(f"{name} ({len(items)}문항): {c}")
print("합계:", tuple(tot), "/ 66")        # v1: (41, 54, 57)

In [ ]:
# Ex14-a2. 근거 선택 평가 함수 (개발셋)
def eval_select(items, model=None):
    H = C = R1 = 0
    wrong = []
    for g in tqdm(items, leave=False):
        found = retrieve(g["q"])[0]                                   # 캐시된 검색 결과
        titles = [c["meta"]["title"] for c in found]
        hit = next((i + 1 for i, t in enumerate(titles) if any(is_hit(t, w) for w in wants_of(g))), None)
        if not hit:
            continue                                                       # 검색이 못 찾은 문항은 제외
        H += 1; R1 += (hit == 1)
        n = select_chunk(g["q"], found, model)
        ok = any(is_hit_loose(titles[n - 1], w) for w in wants_of(g))      # 선택 평가에서만 단수/복수 허용
        C += ok
        if not ok:
            wrong.append((g["q"][:35], g["want"], f"정답 {hit}등", f"고른 것 {n}: {titles[n - 1]}"))
    return H, C, R1, wrong

In [ ]:
# Ex14-b. 점검: 검색 점수 + 선택 호출 속도
for name, items in sets3.items():
    ranks = [rank_in(retrieve(g["q"])[0], g)[0] for g in items]
    print(name, "1등/3등/5등 안:", counts(ranks), "| 후보 개수:", {len(retrieve(g["q"])[0]) for g in items})
print("7B 번역(옛 v1) 기준:", V1_DEV)

g = golden_new[0]; found = retrieve(g["q"])[0]
select_chunk(g["q"], found)                              # 워밍업 (14B를 올리는 시간 제외)
t0 = time.time()
for _ in range(3):
    select_chunk(g["q"], found)
print("선택 호출 1번:", round((time.time() - t0) / 3, 1), "초")      # 1초 안팎이면 정상

In [ ]:
# Ex14-c. 2단계 선택 평가
base14 = {"영어": "20/24", "한국어ko": "18/23", "한국어new": "8/9"}   # 한 번에 하기(14B) 이전 측정
tot = [0, 0, 0]
for name, items in sets3.items():
    H, C, R1, wrong = eval_select(items)
    tot[0] += H; tot[1] += C; tot[2] += R1
    print(f"{name}: 검색 적중 {H} | 2단계 선택 정답 {C}/{H} | (1등 고정 {R1}/{H}, 한 번에 하기 14B {base14[name]})")
    for w in wrong:
        print("   ", w)
print(f"합계: 2단계 {tot[1]}/{tot[0]} | 1등 고정 {tot[2]}/{tot[0]}")

In [ ]:
# Ex14-e. 이웃/형제 제목으로 고른 질문: 고른 근거가 질문에 답이 되는지 직접 읽기
checks = [("숨겨진 걸 찾으려면", golden_ko), ("물속에서 싸우면", golden_ko),
          ("어슴푸레한 숲", golden_ko), ("피해에 저항이", golden_ko), ("약점이 있는 몬스터", golden_ko)]
for key, items in checks:
    g = next(x for x in items if key in x["q"])
    found = retrieve(g["q"])[0]
    n = select_chunk(g["q"], found)
    c = found[n - 1]
    print(f"\nQ. {g['q']}\n정답 제목: {g['want']} | 고른 것: {c['meta']['title']} p.{c['meta']['page']}\n{c['text'][:500]}")

In [ ]:
# Ex15-c. 확인용 실행
test_qs = golden_new[:5] + [g for g in dev if any(w in ("Counterspell", "Light", "Disadvantage") for w in wants_of(g))]
seen = set()
for g in test_qs:
    if g["q"] in seen: continue
    seen.add(g["q"])
    r = explain_chunk(g["q"])
    print(f"\nQ. {g['q']}\n근거: {r['title']} (p.{r['page']}) | 원문 검증 {r['orig_ok']} | 검사 {check_explain(r)}")
    print(f"원문: {r['orig'][:200]}\n설명: {r['explain']}")

In [ ]:
# Ex15-d. 개발셋 38문항 전체: 2단계 답변의 자동 지표
rs = []
for g in tqdm(dev):
    r = explain_chunk(g["q"], temp0=0.0); r["chk"] = check_explain(r); r["q"] = g["q"]; r["g"] = g; rs.append(r)

n = len(rs)
print("원문 검증 통과:", sum(r["chk"]["원문_검증"] for r in rs), "/", n)
print("원문 완결(검증 통과분):", sum(r["chk"]["원문_완결"] for r in rs), "/", sum(r["chk"]["원문_검증"] for r in rs))
print("근거에 없는 숫자:", sum(bool(r["chk"]["근거에_없는_숫자"]) for r in rs), "문항")
print("혼합 철자:", sum(r["chk"]["혼합_철자"] for r in rs), "문항")
print("분수 표현:", sum(bool(r["chk"]["분수"]) for r in rs), "문항")
print("용어 누락:", sum(bool(r["chk"]["용어_누락"]) for r in rs), "문항")
print("깨짐:", sum(is_broken(r["explain"]) for r in rs), "문항")
print("근거 선택 정답(느슨):", sum(any(is_hit_loose(r["title"], w) for w in wants_of(r["g"])) for r in rs), "/", n)
for r in rs:
    if r["chk"]["혼합_철자"] or r["chk"]["분수"] or r["chk"]["근거에_없는_숫자"] or is_broken(r["explain"]):
        print("\n의심:", r["q"], r["chk"], "\n", r["explain"][:200])

In [ ]:
# Ex16-b. (실험) 근거 검증 평가: 선택이 맞으면 '예', 틀리면 '아니오'가 나오는지 / 검색 실패 질문이 거절되는지
tab, detail = Counter(), []
for g in tqdm(dev + golden_ko):
    found, texts = retrieve(g["q"])
    in_top5 = any(is_hit_loose(c["meta"]["title"], w) for c in found for w in wants_of(g))
    n = select_chunk(g["q"], found); c = found[n - 1]
    right = any(is_hit_loose(c["meta"]["title"], w) for w in wants_of(g))
    yes = passes_verify(g["q"], c, texts)          # answer()와 같은 검증
    kind = "선택 맞음" if right else ("선택 틀림" if in_top5 else "검색 실패")
    tab[(kind, "예" if yes else "아니오")] += 1
    if (right and not yes) or (not right and yes):
        detail.append((kind, "예" if yes else "아니오", g["q"][:40], c["meta"]["title"]))

for kind in ("선택 맞음", "선택 틀림", "검색 실패"):
    print(f"{kind}: 예 {tab[(kind,'예')]} | 아니오 {tab[(kind,'아니오')]}")
print("\n[잘못된 거절 / 놓침]")
for d in detail: print(d)

In [ ]:
# Ex16-b2. 거절 정책 비교: A) 아니오면 바로 거절 / B) 다음 후보로 넘어가고 모두 아니오면 거절
def pick_verified(q, found, texts, chain):
    n = select_chunk(q, found)
    order = [n] + [i for i in range(1, len(found) + 1) if i != n]
    for i in (order if chain else [n]):
        if passes_verify(q, found[i - 1], texts):          # answer()와 같은 검증
            return i
    return None                                              # 거절

for name, chain in (("A) 바로 거절", False), ("B) 다음 후보", True)):
    tab = Counter()
    for g in tqdm(dev + golden_ko, desc=name):
        found, texts = retrieve(g["q"])
        in_top5 = any(is_hit_loose(c["meta"]["title"], w) for c in found for w in wants_of(g))
        i = pick_verified(g["q"], found, texts, chain)
        out = "거절" if i is None else ("맞음" if any(is_hit_loose(found[i-1]["meta"]["title"], w) for w in wants_of(g)) else "틀림")
        tab[("검색됨" if in_top5 else "검색 실패", out)] += 1
    print(f"\n{name}")
    for k in ("검색됨", "검색 실패"):
        print(f"  {k}: 맞음 {tab[(k,'맞음')]} | 틀림 {tab[(k,'틀림')]} | 거절 {tab[(k,'거절')]}")

In [ ]:
# Ex16-c. 설명 충실도 검증: 설명에 근거에 없는 내용이 있는지 한 번 더 묻기 (rs = Ex15-d의 결과)
flag = [r for r in tqdm(rs) if not check_faithful(r)]
print("근거에 없는 내용이 있다고 판정:", len(flag), "/", len(rs))
for r in flag:
    print("\nQ.", r["q"], "\n설명:", r["explain"][:300])

r_dv = explain_chunk("How far can a creature with darkvision see?", temp0=0.0)
print("\nDarkvision 설명 충실:", check_faithful(r_dv), "(False면 잡은 것)")

In [ ]:
# Ex16-d. 걸림 비율: 챗봇 모드(재시도 3번)로 개발셋 38문항을 돌렸을 때 얼마나 거절·숨김이 생기나
ask_short("안녕")
stat, why, t0 = Counter(), Counter(), time.time()
for g in tqdm(dev):
    a = answer(g["q"], attempts=CHAT_ATTEMPTS, show_orig=False)
    stat[a["status"]] += 1
    if a["first_flagged"]: stat["첫 시도에 걸림"] += 1
    if a["first_flagged"] and a["status"] == "ok": stat["재시도로 살아남"] += 1
    for w in a["warn"]: why[w.split(" [")[0].split(" (")[0]] += 1
N = len(dev)
print(f"{N}문항 | 질문당 {(time.time()-t0)/N:.1f}초")
print(f"정상 답변 {stat['ok']} | 거절(근거 검증 아니오) {stat['refuse']} | 끝내 숨김 {stat['hidden']}")
print(f"첫 시도에 걸림 {stat['첫 시도에 걸림']} → 재시도로 살아남 {stat['재시도로 살아남']}")
print("끝내 숨긴 이유:", dict(why))
print(f"챗봇이 \"확실하지 않아요\"라고 하는 비율: {(stat['refuse'] + stat['hidden'])}/{N}")

In [ ]:
# Ex18-c. 용어 주입 켜기/끄기 비교 — 개발셋 38문항 (챗봇 모드)

ask_short("안녕")                                   # 워밍업
_terms_for_orig = terms_for                   # 원래 함수를 보관해요

def term_miss_of(a):                          # 원문엔 있는데 설명에 한국어가 없는 용어 목록
    return [ko for ko, en in TERM_MAP.items()
            if re.search(rf"\b{re.escape(en)}\b", a["orig"]) and ko not in a["explain"]]

def run_terms(name, use):
    global terms_for
    terms_for = _terms_for_orig if use else (lambda text: "(없음)")
    stat, miss, saved = Counter(), 0, []
    try:
        for g in tqdm(dev, desc=name):
            a = answer(g["q"], attempts=CHAT_ATTEMPTS, show_orig=False)
            stat[a["status"]] += 1
            if a["status"] == "ok":
                m = term_miss_of(a)
                miss += len(m)
                saved.append({"q": g["q"], "누락": m, "explain": a["explain"]})
    finally:
        terms_for = _terms_for_orig           # 에러가 나도 반드시 원래대로 되돌려요
    json.dump(saved, open(f"{DATA}/dev_terms_{name}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    return name, stat["ok"], stat["refuse"], stat["hidden"], miss

rows = [run_terms("켬", True),]
print(f"{'':4}{'정상':>5}{'거절':>5}{'숨김':>5}{'용어누락':>8}")
for n, ok, rf, hd, ms in rows:
    print(f"{n:4}{ok:>5}{rf:>5}{hd:>5}{ms:>8}")

# 출력 > 켬: 100%
#  38/38 [05:17<00:00,  7.35s/it]
# 끔: 100%
#  38/38 [05:37<00:00,  5.05s/it]
#        정상   거절   숨김    용어누락
# 켬      31    4    3       9
# 끔      28    4    6      18

## 🔒 최종 세트(golden_final, 60문항) — **잠가 둔 셀**

- 최종 세트는 **한 번만** 재는 시험지예요. 개발 중에 점수를 보고 설정을 고르면 안 돼요 (이미 진단 중 일부 설정을 최종 세트로 재서 오염된 적이 있어요. 그 점수로 구성을 고르거나 보고하지 않아요).
- v1 기준 점수는 `final_result_v1.json`에 저장돼 있어요: **31 / 46 / 51** (60문항, 오차 ±10%p 정도).
- 아래 셀들은 `assert`로 잠겨 있어요. 이미 결과 파일이 있으면 에러가 나요(정상).
- v2 측정은 **새로 만든 질문**으로 하거나, 오염을 인정하고 v1과 같은 질문으로 비교하되 그 사실을 같이 적어야 해요.

In [ ]:
# Ex17-a. (선택) v1 재현 검증: 최종 세트 순위가 저장된 v1과 같은지 (0이어야 해요)
# 이 검증은 7B 번역 기준이라 14B 번역에서는 달라지는 게 정상이에요. (참고용)
# saved = {x["q"]: x["rank"] for x in json.load(open(f"{DATA}/final_result_v1.json", encoding="utf-8"))["rows"]}
# diff = [g["q"] for g in tqdm(golden_final) if rank_in(retrieve(g["q"])[0], g)[0] != saved[g["q"]]]
# print("순위가 달라진 문항:", len(diff), "/ 60")

In [ ]:
# Ex17-b. 개발셋과 최종 세트의 정답 겹침 확인 (점수는 재지 않음)
dev_wants = {norm(w) for s in (golden, golden_ko, golden_new) for g in s for w in wants_of(g)}
dup = [(g["q"], w) for g in golden_final for w in wants_of(g) if norm(w) in dev_wants]
print(len(golden_final), "문항 | 개발 세트와 정답이 겹침:", dup)

In [ ]:
# Ex17-pre. 최종셋 캐시 만들기 (속도용, 점수에는 영향 없음). 개발 캐시와 파일을 분리
CACHE_PATH = f"{DATA}/cache_final_14b.json"     # ⚠️ 이 셀 뒤로는 캐시 파일이 최종셋용으로 바뀌어요. 개발셋으로 돌아가려면 Ex00부터 다시
load_cache()
build_cache(golden_final)
save_cache()

In [ ]:
# Ex17-c. 최종 검색 평가 (v1 결과가 이미 있어서 잠겨 있어요. 새 버전을 잴 때만 파일 이름을 바꿔 쓰세요)
final_path = f"{DATA}/final_result_v2.json"
assert not os.path.exists(final_path), "이미 최종 평가를 했어요. 다시 재려면 파일 이름(v2 등)을 바꾸고, 이유를 기록해 두세요."

rows, t0 = [], time.time()
for g in tqdm(golden_final):
    found = retrieve(g["q"])[0]
    r, titles = rank_in(found, g)                 # 정답(want)은 채점에서만 사용
    rows.append({"q": g["q"], "want": g["want"], "rank": r, "top3": titles[:3]})
sec = (time.time() - t0) / len(golden_final)
c = counts([x["rank"] for x in rows])
print(f"최종 세트 (1등 / 3등 안 / 5등 안): {c} / {len(rows)} | 질문당 {sec:.1f}초")
json.dump({"config": "v1 검색 구성(번역① 일반 + 번역② 용어 선택지, 임베딩 3갈래 + BM25 2갈래, 가중 RRF), 번역 모델 14B로 변경 (v1은 7B 번역), num_ctx 4096",
           "counts": c, "sec_per_q": sec, "rows": rows},
          open(final_path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)

In [ ]:
ans_path = f"{DATA}/final_answers_v2.json"
assert not os.path.exists(ans_path), "이미 저장돼 있어요. 새로 만들 때는 다른 파일 이름을 쓰세요."
answers = []
for g in tqdm(golden_final):
    a = answer(g["q"], attempts=CHAT_ATTEMPTS, show_orig=False)       # 챗봇 모드와 같은 설정
    answers.append({"q": g["q"], "status": a["status"], "warn": a["warn"],
                    "title": a.get("title"), "page": a.get("page"),         # 모델이 고른 근거 (정답 제목 아님)
                    "orig": a.get("orig"), "body": a.get("body"),            # 판정자가 대조할 근거
                    "explain": a.get("explain"),                             # 숨긴 설명도 저장 (막은 게 맞았는지 보려고)
                    "candidates": a.get("candidates")})
json.dump(answers, open(ans_path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print(len(answers), "개 저장 |", Counter(x["status"] for x in answers))

## 정리 체크리스트

**A. 재현성**: `GEN_SEED=42`, 평가는 `temperature 0`, 챗봇 재시도만 seed를 바꿈. 같은 문항이라도 ±1문항 차이는 잡음으로 봐요. 모델을 올린 첫 호출은 워밍업. `OLLAMA_KEEP_ALIVE=1h`로 중간에 내려가는 것 방지.

**B. 평가**: 개발셋(38)은 디버깅·회귀용(오염됨), 새 세트는 만들지 않기로 함. golden_final은 v2로 **한 번만** 재고 오염 사실을 같이 적기. 지표는 자동 검사만(깨짐·혼합·분수·숫자·원문 검증), 용어 누락은 참고용.

**C. 용어표**: Ex09-a 코드 안에 고정 (상태 항목은 삭제). 결과를 보고 용어를 추가하지 않기. 역할은 한국어 용어 통일.

**D. 파이프라인**: 선택 → 검증/거절(정책 A, 제목이 번역과 일치하면 검증 생략) → 설명 → 규칙 검사 → 충실성. 알려진 약점: Hide/Grappled류의 미묘한 의미 변화(분수 뒤집힘, 한정어 누락)는 못 잡을 수 있음.

**E. 배포**: 14B ↔ bge-m3 교체 시간(턴당 ~13초), 입력 분류기, 시나리오 RAG, FastAPI+ngrok. 챗봇 화면의 출처 표시는 팀 UX 결정.

**F. 한계**: 사용자가 원문으로 확인할 수 없으면 검사가 못 잡는 오류는 그대로 나감 → Ex16-d의 거절 비율과 함께 팀에 공유.

In [ ]:
import csv
rows_out = []
for n, x in enumerate(json.load(open(ans_path, encoding="utf-8")), 1):
    rows_out.append([n, x["q"], x["status"], "; ".join(x["warn"]),
                     f"{x['title']} p.{x['page']}" if x.get("title") else "",
                     x.get("orig") or "", x.get("explain") or "",
                     "; ".join(f"{t} p.{p}" for t, p in (x.get("candidates") or [])), "", ""])
jp = f"{DATA}/judge_v2.csv"
with open(jp, "w", newline="", encoding="utf-8-sig") as f:       # utf-8-sig: 엑셀에서 한글이 안 깨져요
    w = csv.writer(f)
    w.writerow(["번호", "질문", "상태", "숨김 사유", "고른 근거", "원문", "설명", "거절 시 후보", "판정", "메모"])
    w.writerows(rows_out)
files.download(jp)

## 부록. 이전 작업·테스트 기록

### ♻️ v4에서 옮긴 옛 실험 셀 (전부 주석, 옛 함수 이름 그대로)

In [ ]:
# Ex09-c (v3). 옛 답변 프롬프트 RAG_TEMPLATE
# Ex09-c. 답변 프롬프트
# RAG_TEMPLATE = """당신은 D&D 규칙 안내자입니다. 아래 규칙을 반드시 지키세요.
# 1. [근거]는 번호가 붙은 여러 개입니다. 질문에 가장 직접 답하는 근거 하나를 고르세요. 관련 없는 근거는 무시하세요.
# 2. 고른 근거에 적힌 내용만 사용하세요. 어느 근거에도 없으면 "근거에서 찾지 못했어요"라고만 답하세요.
# 3. 용어는 이 표의 한국어를 쓰고 영어 원문을 괄호로 함께 적으세요: {terms}
# 4. 근거의 조건, 숫자, 이점/불리점을 빼거나 바꾸지 마세요.
# 5. 답변은 "근거 번호:", "원문:", "설명:" 세 줄로 쓰세요. 원문은 고른 근거에서 질문에 직접 답하는 문장을 영어 그대로 복사하고, 설명은 한국어로 쓰세요.

# ### 예시
# 질문: What does the Dash action do?
# 근거:
# [1] (Torch (1 CP) p.100) A Torch burns for 1 hour, casting Bright Light in a 20-foot radius and Dim Light for an additional 20 feet.
# [2] (Dash [Action] p.181) When you take the Dash action, you gain extra movement for the current turn. The increase equals your Speed after applying any modifiers.
# 답변:
# 근거 번호: 2
# 원문: When you take the Dash action, you gain extra movement for the current turn. The increase equals your Speed after applying any modifiers.
# 설명: 질주(Dash) 행동을 하면 이번 턴에 속도(Speed)만큼 이동 거리가 늘어납니다.

# ### 근거
# {context}

# ### 질문
# {question}

# ### 답변"""


In [ ]:
# Ex10. 질문 하나 → 검색 → 답변. 챗봇 본체에서 이 함수를 부름
# def rag_answer(question, k=5, tries=3):
#     found, texts = search_final(question, k)
#     context = "\n".join(f"[{n}] ({c['meta']['title']} p.{c['meta']['page']}) {c['text'][:2000]}"
#                         for n, c in enumerate(found, 1))
#     prompt = RAG_TEMPLATE.format(terms=TERMS, context=context, question=question)
#     for t in range(1, tries + 1):                      # 한자·일본어·베트남어 등이 섞이면 temperature를 올려 다시 생성
#         answer = ask_llm(prompt, temperature=0.1 + 0.3 * (t - 1))
#         if not is_broken(answer):
#             break
#     sources = [f"{c['meta']['title']} (p.{c['meta']['page']})" for c in found]
#     return {"answer": answer, "sources": sources, "queries": texts[1:], "tries": t}

# def ask_rules(q, answer=True):                         # 노트북에서 직접 확인하는 용도
#     if not answer:
#         found, texts = search_final(q)
#         print("번역:", texts[1], "|", texts[2])
#         print("검색:", [f"{c['meta']['title']} (p.{c['meta']['page']})" for c in found])
#         return
#     r = rag_answer(q)
#     print("번역:", r["queries"][0], "|", r["queries"][1])
#     print("검색:", r["sources"])
#     print("\n" + r["answer"])

In [ ]:
# Ex11. 검색 결과를 텍스트 파일로 저장해서 눈으로 확인 (✅ = 정답 제목과 일치)
# def save_retrieval_check(items, path):
#     with open(path, "w", encoding="utf-8") as f:
#         for n, g in enumerate(items, 1):
#             found, texts = search_final(g["q"])
#             f.write(f"########## Q{n}. {g['q']}\n번역: {texts[1]} | {texts[2]}\n정답 제목: {g['want']}\n\n")
#             for r, c in enumerate(found, 1):
#                 ok = any(is_hit(c["meta"]["title"], w) for w in wants_of(g))
#                 f.write(f"--- {r}등 {'✅정답' if ok else ''} | {c['meta']['title']} | p.{c['meta']['page']} | {c['id']}\n")
#                 f.write(c["text"][:400].replace("\n", " ") + "\n\n")
#             f.write("\n")

# CHECK_PATH = f"{DATA}/retrieval_check.txt"
# save_retrieval_check(golden_new, CHECK_PATH)
# files.download(CHECK_PATH)

In [ ]:
# Ex13-a. 개발셋 정답 청크 매칭
# def gold_chunks(g, limit=3):       # want 제목과 일치하는 청크를 찾고, 질문당 최대 limit개만 사용
#     hits = [c for c in chunks
#             if any(is_hit(c["meta"]["title"], w) for w in wants_of(g))]
#     return hits[:limit]

# for g in dev:
#     g["gold"] = gold_chunks(g)

# print(len(dev), "문항")
# print("정답 청크 0개인 문항:", [g["q"] for g in dev if not g["gold"]])
# print("청크 수 분포:", [len(g["gold"]) for g in dev])

In [ ]:
# Ex13-b (v3). 설명 품질 평가 도구 — 정답 청크(g['gold'])가 필요해서 Ex13-a와 함께만 써요
# Ex13-b. 평가 도구
# def build_ctx(g):                                   # 정답 청크를 번호 붙여 근거로 만들기
#     return "\n".join(f"[{n}] ({c['meta']['title']} p.{c['meta']['page']}) {c['text'][:2000]}"
#                      for n, c in enumerate(g["gold"], 1))

# def expl_part(answer):                              # "설명:" 뒤만 평가 (근거 번호·원문 줄은 제외)
#     m = re.search(r"설명\s*[:：]\s*(.*)", answer, flags=re.S)
#     return m.group(1).strip() if m else answer

# NUM = re.compile(r"\d+(?:d\d+)?", re.I)            # 숫자 또는 주사위 표기(2d6)
# def numbers(text):
#     return [n.lower() for n in NUM.findall(text)]

# def src_text(g):
#     return " ".join(c["text"] for c in g["gold"])

# 영어 숫자 단어 → 숫자 (주의: "Three-Quarters" 같은 분수를 숫자로 풀어 버려서 "3분의 4" 같은 뒤집힘 오류를 놓칠 수 있어요)
# W2N = {"one": "1", "two": "2", "three": "3", "four": "4", "five": "5", "six": "6", "seven": "7",
#        "eight": "8", "nine": "9", "ten": "10", "half": "1 2", "quarter": "1 4", "quarters": "4"}
# W2N.update({"twice": "2", "double": "2", "doubled": "2"})      # 두 배 표현도 숫자 2로 봐요

# def src_numbers(g):
#     s = src_text(g).lower()
#     extra = [d for w in re.findall(r"[a-z]+", s) if w in W2N for d in W2N[w].split()]
#     return set(numbers(s)) | set(extra)

# def score(rows, items=None):                         # 간단 요약: 깨짐 / 누출 / 숫자 충실도
#     items = items or dev
#     broken, leak, tot, ok, bad_q = [], [], 0, 0, []
#     for r, g in zip(rows, items):
#         e = expl_part(r["answer"])
#         if is_broken(e): broken.append(g["q"])
#         if LEAK.search(e): leak.append(g["q"])
#         src, nums = src_numbers(g), numbers(e)
#         good = sum(n in src for n in nums); tot += len(nums); ok += good
#         if good < len(nums): bad_q.append((g["q"], [n for n in nums if n not in src]))
#     print(f"깨짐 {len(broken)}/{len(rows)} | 대화턴 누출 {len(leak)} | 숫자 충실도 {ok}/{tot} | 근거에 없는 숫자 문항 {len(bad_q)}")
#     return broken, bad_q

# norm_txt = lambda s: re.sub(r"\s+", " ", re.sub(r"[\"'“”‘’]", "", s)).strip().lower()
# TERM_REV = {en.lower(): ko for ko, en in TERM_MAP.items()}        # 영어 → 한국어

# def eval_expl(name, rows, items=None):               # 자세한 자동 채점 (문항별 기준을 손으로 쓰지 않아요)
#     items = items or dev
#     n_orig = copy_ok = nums_t = nums_ok = term_t = term_ok = 0
#     mism = []
#     for r, g in zip(rows, items):
#         ans = r["answer"]
#         orig = " ".join(re.findall(r"원문\s*[:：]\s*(.+)", ans))      # 모델이 고른 영어 문장
#         e = expl_part(ans)
#         if orig:
#             n_orig += 1
#             copy_ok += norm_txt(orig) in norm_txt(src_text(g))        # 근거 안에 실제로 있는 문장?
#         o_nums, e_nums = numbers(orig), set(numbers(e))
#         nums_t += len(o_nums); nums_ok += sum(n in e_nums for n in o_nums)
#         for ko, en in TERM_MAP.items():                               # 원문의 용어표 단어 → 한국어가 설명에 있나
#             if re.search(rf"\b{re.escape(en)}\b", orig, re.I):
#                 term_t += 1; term_ok += ko in e
#         for m in re.finditer(r"\(([A-Za-z][A-Za-z \-]+)\)", e):       # 한글(English) 쌍이 용어표와 맞나
#             ko = TERM_REV.get(m.group(1).strip().lower())
#             if ko and ko not in e[max(0, m.start() - 20):m.start()]:
#                 mism.append((g["q"][:25], ko, m.group(1)))
#     broken = sum(is_broken(expl_part(r["answer"])) for r in rows)
#     leak = sum(bool(LEAK.search(expl_part(r["answer"]))) for r in rows)
#     print(f"{name:12s} 깨짐 {broken} | 누출 {leak} | 원문 복사 {copy_ok}/{n_orig} | 숫자 보존 {nums_ok}/{nums_t} | "
#           f"용어 보존 {term_ok}/{term_t} | 용어 불일치 {len(mism)}")
#     return mism

In [ ]:
# Ex13-c. 좁힌 프롬프트 (비교용, 채택 안 함) + 변형 실행기
# NARROW_TEMPLATE = """당신은 D&D 규칙을 한국어로 옮기는 번역가입니다. 아래 규칙을 반드시 지키세요.
# 1. [근거]는 번호가 붙은 영어 규칙입니다. 질문에 가장 직접 답하는 근거 하나를 고르세요.
# 2. 고른 근거에 적힌 내용만 한국어로 옮기거나 요약하세요. 근거에 없는 내용, 추측, 일반 상식을 덧붙이지 마세요.
# 3. 숫자, 주사위 표기(예: 2d6), 거리·시간 단위, DC, +2 같은 보너스는 근거에 적힌 그대로 쓰세요. 바꾸거나 계산하지 마세요.
# 4. 규칙 용어는 이 표의 한국어를 쓰고 영어를 괄호로 함께 적으세요. 표에 없는 용어는 영어 그대로 두세요: {terms}
# 5. 한글, 영어 알파벳, 숫자만 사용하세요. 한자, 일본어, 베트남어는 쓰지 마세요.
# 6. 답변은 "근거 번호:", "원문:", "설명:" 세 줄로 쓰세요. 원문은 고른 근거에서 질문에 직접 답하는 문장을 영어 그대로 복사하세요.

### 예시
# 질문: What does the Dash action do?
# 근거:
# [1] (Torch (1 CP) p.100) A Torch burns for 1 hour, casting Bright Light in a 20-foot radius and Dim Light for an additional 20 feet.
# [2] (Dash [Action] p.181) When you take the Dash action, you gain extra movement for the current turn. The increase equals your Speed after applying any modifiers.
# 답변:
# 근거 번호: 2
# 원문: When you take the Dash action, you gain extra movement for the current turn. The increase equals your Speed after applying any modifiers.
# 설명: 질주(Dash) 행동을 하면 이번 턴에 이동 거리가 늘어납니다. 늘어나는 양은 수정치를 적용한 속도(Speed)와 같습니다.

### 근거
# {context}

### 질문
# {question}

### 답변"""

# def run_variant(name, model, template, items=None):
#     items = items or dev
#     path = f"{DATA}/expl_{name}.json"
#     if os.path.exists(path):
#         print("캐시 사용:", name)
#         return json.load(open(path, encoding="utf-8"))["rows"]
#     g0 = items[0]
#     gen(template.format(terms=TERMS, context=build_ctx(g0), question=g0["q"]), model)   # 워밍업: 실제 프롬프트로 (결과 버림)
#     rows, t0 = [], time.time()
#     for g in tqdm(items, desc=name):
#         prompt = template.format(terms=TERMS, context=build_ctx(g), question=g["q"])
#         rows.append({"q": g["q"], "answer": gen(prompt, model)})
#     sec = (time.time() - t0) / len(items)
#     json.dump({"model": model, "sec_per_q": sec, "rows": rows},
#               open(path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
#     print(f"{name}: 질문당 {sec:.1f}초")
#     return rows

In [ ]:
# Ex13-d. 4가지 변형 결과 비교 (저장된 파일이 있으면 바로 불러와요. 없으면 새로 생성)
# res = {}
# for _k, _n, _t in (("A_7B_현행", "A_7B_current", RAG_TEMPLATE), ("B_7B_좁힘", "B_7B_narrow", NARROW_TEMPLATE)):
#     if os.path.exists(f"{DATA}/expl_{_n}.json"):        # 7B는 예전에 저장한 결과가 있을 때만 비교에 넣어요 (새로 안 돌려요)
#         res[_k] = run_variant(_n, "qwen2.5:7b", _t)
# res["C_14B_현행"] = run_variant("C_14B_current", "qwen2.5:14b", RAG_TEMPLATE)    # ← 채택
# res["D_14B_좁힘"] = run_variant("D_14B_narrow",  "qwen2.5:14b", NARROW_TEMPLATE)

# for k, rows in res.items():
#     print(k); score(rows)
# print()
# mism_all = {k: eval_expl(k, rows) for k, rows in res.items()}
# for k, m in mism_all.items():
#     print(k, m[:6])

In [ ]:
# Ex13-e. 14B 현행(C) 답변에서 의심 문항 직접 보기 (깨짐 / 숫자가 근거에 없음 / 용어 불일치)
# rows = res["C_14B_현행"]
# for r, g in zip(rows, dev):
#     e = expl_part(r["answer"])
#     bad = sorted(set(BROKEN.findall(e)))
#     extra = [n for n in numbers(e) if n not in src_numbers(g)]
#     if bad or extra:
#         print(f"[깨진 글자 {bad} | 근거에 없는 숫자 {extra}] {g['q']}\n   {e[:300]}\n")

In [ ]:
# Ex14-d. 한 번에 하기 방식으로 근거를 제대로 고르는지 (옛 평가, 14B 현행 rag_answer 사용)
# def pick_no(answer):
#     m = re.search(r"근거 번호\s*[:：]\s*(\d+)", answer)
#     return int(m.group(1)) if m else None

# def eval_pick(items):
#     rows = []
#     for g in tqdm(items):
#         r = rag_answer(g["q"])
#         titles = [re.sub(r"\s*\(p\.\d+\)$", "", s) for s in r["sources"]]
#         hit_rank = next((i + 1 for i, t in enumerate(titles) if any(is_hit(t, w) for w in wants_of(g))), None)
#         n = pick_no(r["answer"])
#         ok = n is not None and 1 <= n <= len(titles) and any(is_hit(titles[n - 1], w) for w in wants_of(g))
#         rows.append({"q": g["q"], "want": g["want"], "hit_rank": hit_rank, "picked": n,
#                      "picked_title": titles[n - 1] if n and 1 <= n <= len(titles) else None, "ok": ok})
#     return rows

# 오래 걸려요(질문마다 14B 생성). 필요할 때만 주석을 풀어 실행하세요.
# all_rows = {name: eval_pick(items) for name, items in sets3.items()}
# for name, rows in all_rows.items():
#     H = [x for x in rows if x["hit_rank"]]; C = [x for x in H if x["ok"]]
#     print(f"{name}: 검색 적중 {len(H)}/{len(rows)} | 근거 선택 정답 {len(C)}/{len(H)}")

In [ ]:
# Ex15. 속도 점검: 캐시 뒤 질문당 시간 (6초 안팎이어야 해요. 전에는 18~20초)
# rag_answer(golden_new[0]["q"])                         # 워밍업: 14B 올리는 시간은 빼고 재요
# t0 = time.time()
# for g in golden_new[1:4]:
#     r = rag_answer(g["q"])
# print("질문당", round((time.time() - t0) / 3, 1), "초")
# !ollama ps
# 질문당 4.8 초
# NAME           ID              SIZE     PROCESSOR    CONTEXT    RUNNER      UNTIL
# qwen2.5:14b    7cdf5a0187d5    10 GB    100% GPU     8192       llamacpp    59 minutes from now

## 참고: 재순위(rerank) 실험 — 채택하지 않음
`bge-reranker-v2-m3`로 후보 20개를 다시 정렬해 봤지만 1등을 뺏기는 질문이 생겨서 쓰지 않아요. 기록용으로만 남겨 둬요 (실행하려면 주석 해제).

In [ ]:
# Ex16. 재순위 실험 (참고용, 기본은 실행 안 함). 후보 20개 -> 재순위 -> 5개
# !pip install -q sentence-transformers
# from sentence_transformers import CrossEncoder
# reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512, device="cuda")
#
# def rerank_fn(use="ko", pool_k=20, k=5):
#     def fn(q):
#         pool, texts = search_final(q, pool_k)       # texts = [원문, 번역1, 번역2]
#         query = q if use == "ko" else texts[1]
#         scores = reranker.predict([(query, c["text"][:1500]) for c in pool], batch_size=16)
#         order = sorted(range(len(pool)), key=lambda i: scores[i], reverse=True)
#         return [pool[i] for i in order[:k]]
#     return fn
#
# variants = {"기준(재순위 없음)": lambda q: search_final(q)[0],
#             "R1 재순위(한국어 질문)": rerank_fn("ko"),
#             "R2 재순위(영어 번역)":   rerank_fn("en")}
# for name, fn in variants.items():
#     r = {s: evaluate(fn, items) for s, items in sets3.items()}
#     print(f"{name:20s}", r, "| 합계(66):", tuple(sum(v[i] for v in r.values()) for i in range(3)))

### 📦 데이터 준비 · PDF → 청크 만들기 (글꼴 크기로 제목 단계 판별)
- PDF 361쪽을 읽어 청크 1958개 생성 후 `rules_new.jsonl`로 저장. 이후 이 노트북은 파일만 읽어서 1957개를 사용
- 01_rules_data 노트북 단계의 작업이라 본문에서 뺐어요
- 출처: `TRPG_Rule` 셀 8, `TRPG_Rule` 셀 9, `TRPG_Rule` 셀 10, `TRPG_Rule` 셀 11, `TRPG_Rule` 셀 12, `TRPG_Rule` 셀 13, `TRPG_Rule` 셀 14, `TRPG_Rule` 셀 15, `TRPG_Rule` 셀 16, `TRPG_Rule` 셀 17, `TRPG_Rule` 셀 18, `TRPG_Rule` 셀 19

In [ ]:
# ───── 원본: TRPG_Rule · 셀 8 ─────
# PDF열기 + 원본 확인
# import pymupdf
#
# doc = pymupdf.open(PDF_PATH)
# print("전체 페이지 수:", len(doc))
# print("메타데이터:", doc.metadata)
#
# print(doc[0].get_text("text")[:300])    # 0번 = 첫 페이지
#
# for page_no in range(10):
#     if "Creative Commons" in doc[page_no].get_text("text"):
#         print("CC 표기 발견: p.", page_no + 1)

# ───── 원본: TRPG_Rule · 셀 9 ─────
# 페이지 하나를 통쨰로 보기
# PAGE_NO = 20
# print(doc[PAGE_NO - 1].get_text("text")[:1500])

# ───── 원본: TRPG_Rule · 셀 10 ─────
# 글꼴 크기, 종류별 통계
# from collections import Counter
#
# stat = Counter()            # {(글꼴 크기, 글꼴 이름): 글자 수}
# example = {}                # {(글꼴 크기, 글꼴 이름): 예시 문장}
#
# for page in doc:
#     data = page.get_text("dict")                # 글꼴 정보(dict로 확인)까지 포함
#     for block in data["blocks"]:
#         for line in block.get("lines", []):     # 이미지 블록에는 lines가 없어요
#             for span in line["spans"]:          # span : 같은 글꼴로 이어진 글자 조각
#                 text = span["text"].strip()
#                 if not text:
#                     continue
#                 key = (round(span["size"], 1), span["font"])
#                 stat[key] += len(text)
#                 example.setdefault(key, text[:40])      # setdefault : 키가 없을 때의 값
#
# for key in sorted(stat, reverse=True):
#     size, font = key
#     print(f"{size:5} | {font:28} | 글자 수 {stat[key]:7,} | 예: {example[key]}")

# ───── 원본: TRPG_Rule · 셀 11 ─────
# 줄 단위로 읽는 helper + 제목 후보 목록
# 글꼴 정보와 글을 꺼내는 helper
# def iter_lines(page):
#     for block in page.get_text("dict")["blocks"]:
#         for line in block.get("lines", []):
#             spans = [s for s in line["spans"] if s["text"].strip()]
#             if not spans:
#                 continue
#             styles = {(round(s["size"], 1), s["font"]) for s in spans}  # 집합 컴프리헨션 {… for s in spans}: 리스트 컴프리헨션에 [ ] 대신 { }를 쓰면 중복이 사라짐
#             text = "".join(s["text"] for s in spans).strip()
#             yield styles, text, line["bbox"]        # yield: return처럼 값을 내놓되 함수가 끝나지 않고, for로 하나씩 꺼내 쓸 수 있는 방식
#
# HEAD_CANDIDATES = [
#     (26.0, "GillSans-SemiBold"),
#     (18.0, "GillSans-SemiBold"),
#     (14.8, "GillSans-SemiBold"),
#     (14.0, "GillSans-SemiBold"),
#     (12.0, "GillSans-SemiBold"),
#     (10.5, "GillSans-SemiBold"),
# ]
#
# found = {style: [] for style in HEAD_CANDIDATES}      # {글꼴: [(쪽, 글), ...]}
#
# for page_no, page in enumerate(doc, start=1):
#     for styles, text, bbox in iter_lines(page):
#         if len(styles) != 1:                          # 글꼴이 섞인 줄은 제목이 아님
#             continue        # len(styles) != 1이면 건너뜀: 한 줄에 글꼴이 섞이면 본문 속 강조 표현이지 제목이 아님
#         style = list(styles)[0]
#         if style in found:
#             found[style].append((page_no, text))
#
# for style in HEAD_CANDIDATES:
#     items = found[style]
#     print(f"\n{style} | 줄 수: {len(items)}")
#     for page_no, text in items[:6] + items[-6:]:      # 앞 6개 + 뒤 6개
#         print("   p.", page_no, "|", text)

# ───── 원본: TRPG_Rule · 셀 12 ─────
# 머리글, 쪽 번호 확인
# HEADER_TEXT = "System Reference Document 5.2"
#
# header_count = 0
# pagenum_count = 0
# for page in doc:
#     for styles, text, bbox in iter_lines(page):
#         if text == HEADER_TEXT:
#             header_count += 1
#         elif text.isdigit() and (11.0, "GillSans-SemiBold") in styles:
#             pagenum_count += 1
#
# print("머리글 줄 수:", header_count)          # 361이면 쪽마다 1개
# print("쪽 번호 줄 수:", pagenum_count)        # 361이면 쪽마다 1개

# ───── 원본: TRPG_Rule · 셀 13 ─────
# Ex10. 정답 제목이 어느 크기에 있는지 보기
#
# golden = json.load(open(f"{DATA}/golden.json", encoding="utf-8"))
#
# want는 문자열이거나 리스트라서, 전부 하나의 리스트로 통일
# wants = []
# for g in golden:
#     w = g["want"]
#     wants += [w] if isinstance(w, str) else w
#
# for want in wants:
#     exact = []                    # 글자가 완전히 같은 제목
#     partial = []                  # 일부만 겹치는 제목
#     for style in HEAD_CANDIDATES:
#         for page_no, text in found[style]:
#             if text == want:
#                 exact.append((style[0], page_no))
#             elif want in text:
#                 partial.append((style[0], page_no, text))
#     print(f"{want:34} | 정확히 일치: {exact[:3]} | 부분 일치 {len(partial)}건")
#     if not exact and partial:
#         print("      부분 예:", partial[:2])

# ───── 원본: TRPG_Rule · 셀 14 ─────
# Ex11. 제목 크기 → 단계, 청크 만들기
# import re
#
# FONT = "GillSans-SemiBold"
# LEVEL = {                                   # (크기, 글꼴) → 단계
#     (26.0, FONT): 1,                        # 장
#     (18.0, FONT): 2,                        # 절 / 몬스터
#     (14.0, FONT): 3,                        # 소절
#     (12.0, FONT): 4,                        # 항목·용어·주문
#     (14.8, FONT): 5,                        # 스탯블록 이름 (가장 깊은 단계)
# }                                           # 10.5(표 제목)는 일부러 뺌 → 본문 취급
# PAGE_NUM_STYLE = (11.0, FONT)
# STAT = re.compile(r"\b(Str|Dex|Con|Int|Wis|Cha)\s*(\d{1,2})\s*([+\-−]\d+)\s*([+\-−]\d+)")
#
# BODY_TITLE = "Playing the Game"             # 본문 첫 장 (그 앞 2~4쪽은 목차)
# BODY_START = [p for p, t in found[(26.0, FONT)] if t == BODY_TITLE][0]   # Ex08 결과 재사용 → 5
#
# def iter_lines_b(page):
#     """iter_lines + 블록 번호 (같은 블록 = 같은 문단/제목)"""
#     for b_no, block in enumerate(page.get_text("dict")["blocks"]):
#         for line in block.get("lines", []):
#             spans = [s for s in line["spans"] if s["text"].strip()]
#             if not spans:
#                 continue
#             styles = {(round(s["size"], 1), s["font"]) for s in spans}
#             text = "".join(s["text"] for s in spans).strip()
#             yield styles, text, b_no
#
# def clean_join(lines):
#     """줄 목록 → 한 문단. 탭 제거, 줄 끝 하이픈 이어붙이기"""
#     out = ""
#     for ln in lines:
#         ln = ln.replace("\t", " ").strip()
#         if not ln:
#             continue
#         if out.endswith("-") and ln[0].islower():   # proficien- + cies
#             out = out[:-1] + ln
#         else:
#             out = out + " " + ln if out else ln
#     return re.sub(r" {2,}", " ", out)
#
# def fix_stat(body):
#     """Str18+4 +4 → Str 18 (mod +4, save +4), MODSAVE 라벨 제거"""
#     body = STAT.sub(r"\1 \2 (mod \3, save \4)", body)
#     return re.sub(r"(?:MODSAVE\s*)+", "", body)
#
# chunks = []        # 완성된 청크
# dropped = []       # 본문이 없어 버린 제목
# merged = []        # 두 줄 제목 합친 기록
# unconv = []        # 스탯 변환 실패 기록
# pre_lines = []     # 장 제목보다 먼저 읽힌 같은 쪽 본문 기록
# path = {}          # {단계: 제목} 지금 위치
#
# def open_chunk(title, level, page_no, b_no):
#     for k in [k for k in path if k >= level]:
#         del path[k]
#     crumb = [path[k] for k in sorted(path) if k >= 1]
#     path[level] = title
#     return {"title": title, "level": level, "crumb": crumb, "block": b_no,
#             "page": page_no, "page_end": page_no, "lines": []}
#
# def close_chunk(c):
#     raw = clean_join(c["lines"])
#     body = fix_stat(raw)
#     if not body:
#         dropped.append((c["page"], c["title"]))
#         return
#     if "MODSAVE" in raw and "(mod " not in body:
#         unconv.append((c["page"], c["title"]))
#     chain = c["crumb"] + [c["title"]]
#     chunks.append({
#         "id": f"rule_{len(chunks) + 1:04d}",
#         "text": " > ".join(chain) + "\n" + body,
#         "meta": {
#             "type": "meta" if chain[0] == "Legal Information" else "rule",   # 약관은 보존만
#             "title": c["title"],
#             "section": chain[1] if len(chain) > 1 else chain[0],
#             "h1": chain[0],
#             "page": c["page"],
#             "page_end": c["page_end"],
#             "source": "SRD_CC_v5.2",
#         },
#     })
#
# cur = open_chunk("Front Matter", 0, 1, -1)
#
# for page_no, page in enumerate(doc, start=1):
#     if 1 < page_no < BODY_START:            # 목차 쪽(2~4)은 통째로 건너뜀
#         continue
#     if page_no == BODY_START:               # 박스 글이 장 제목보다 먼저 읽히므로 장을 미리 열어 둠
#         close_chunk(cur)
#         cur = open_chunk(BODY_TITLE, 1, page_no, -1)
#     start_id, start_len = id(cur), len(cur["lines"])   # 이 쪽 시작 시점 기록
#
#     for styles, text, b_no in iter_lines_b(page):
#         style = list(styles)[0] if len(styles) == 1 else None
#
#         # 1) 머리글·쪽 번호 / 2) 목차 점선 줄 버리기
#         if text == HEADER_TEXT or (style == PAGE_NUM_STYLE and text.isdigit()):
#             continue
#         if re.search(r"\.{4,}", text):
#             continue
#
#         level = LEVEL.get(style)
#
#         # 3) 제목이 아니면 본문
#         if level is None:
#             cur["lines"].append(text)
#             cur["page_end"] = page_no
#             continue
#
#         # 4) 미리 열어 둔 장의 진짜 제목이 뒤늦게 나오면 무시
#         if level == 1 and text == cur["title"]:
#             continue
#
#         # 5) 두 줄로 쪼개진 제목 합치기 (장 제목은 블록이 달라도 합침)
#         if (level == cur["level"] and not cur["lines"] and cur["page"] == page_no
#                 and (level == 1 or b_no == cur["block"])):
#             merged.append((cur["title"], text))
#             cur["title"] += " " + text
#             path[level] = cur["title"]
#             continue
#
#         # 6) 스탯블록 안에서 몬스터 이름이 한 번 더 찍힌 줄 건너뛰기
#         if style == (14.8, FONT) and text == path.get(2):
#             continue
#
#         # 7) 새 장 제목: 같은 쪽에서 제목보다 먼저 읽힌 박스 글은 앞 청크에서 떼어 새 장으로 옮김
#         carry = []
#         if level == 1 and id(cur) == start_id and len(cur["lines"]) > start_len:
#             carry = cur["lines"][start_len:]
#             del cur["lines"][start_len:]
#             cur["page_end"] = max(cur["page"], page_no - 1)
#             pre_lines.append((page_no, text, len(carry)))
#         close_chunk(cur)
#         cur = open_chunk(text, level, page_no, b_no)
#         cur["lines"] = carry
#
# close_chunk(cur)
# print("청크 수:", len(chunks), "| 본문 없어 버린 제목:", len(dropped), "| 두 줄 제목 합침:", len(merged))
# print("스탯 변환된 청크:", sum("(mod " in c["text"] for c in chunks),
#       "| MODSAVE 남은 청크:", sum("MODSAVE" in c["text"] for c in chunks))
# print("스탯 변환 실패:", unconv)
# print("장 제목 앞에 같은 쪽 본문이 먼저 붙은 경우:", pre_lines)
# for a, b in merged[:15]:
#     print("  합침:", a, "+", b)
#
# 변환 안 된 능력치가 남은 청크 수 (정상이면 0)
# left = [c["id"] for c in chunks
#         if re.search(r"\b(?:Str|Dex|Con|Int|Wis|Cha)\s*\d{1,2}\s*[+\-−]\d+\s*[+\-−]\d+", c["text"])]
# print("미변환 능력치가 남은 청크:", len(left), left[:5])

# ───── 원본: TRPG_Rule · 셀 15 ─────
# Ex11b. 빼기 전에 눈으로 확인
# (1) 머리글 문구가 남은 청크
# for c in chunks:
#     if HEADER_TEXT in c["text"]:
#         i = c["text"].index(HEADER_TEXT)
#         print("머리글 잔존:", c["id"], c["meta"]["h1"], "p.", c["meta"]["page"],
#               "|", repr(c["text"][max(0, i - 60):i + 60]))
#
# (2) 뺄 후보 3개의 내용
# for c in chunks:
#     if c["meta"]["h1"] in {"Legal Information", "Contents", "Index of Stat Blocks"}:
#         print("\n##", c["meta"]["h1"], "| 길이", len(c["text"]),
#               "| p.", c["meta"]["page"], "-", c["meta"]["page_end"])
#         print(c["text"][:400])
#
# (3) 본문이 없어 버려진 제목
# print("\n버린 제목:", len(dropped))
# print(", ".join(f"{p}:{t}" for p, t in dropped[:60]))
#
# (4) MODSAVE 위치 + 스탯블록 하나 전체
# print("\nMODSAVE 포함 청크:", Counter(c["meta"]["h1"] for c in chunks if "MODSAVE" in c["text"]))
# hag = [c for c in chunks if c["meta"]["title"] == "Green Hag"][0]
# print("\n", hag["text"][:1500])

# ───── 원본: TRPG_Rule · 셀 16 ─────
# Ex11c. 1~5쪽에 뭐가 있는지
# print(found[(26.0, FONT)])                    # 장 제목 26.0 줄이 어느 쪽에 있나
# for p in (2, 3, 5):
#     print(f"\n--- p.{p}")
#     for styles, text, bbox in list(iter_lines(doc[p - 1]))[:16]:
#         tag = "혼합" if len(styles) > 1 else str(list(styles)[0][0])
#         print(f"{tag:>5} | {text[:70]}")

# ───── 원본: TRPG_Rule · 셀 17 ─────
# Ex11d. 장 제목 앞에 붙은 줄이 어느 쪽 내용인지 확인
# for p in (89, 104):
#     print(f"\n--- p.{p}")
#     for styles, text, b_no in list(iter_lines_b(doc[p - 1]))[:16]:
#         tag = "혼합" if len(styles) > 1 else str(list(styles)[0][0])
#         print(f"{tag:>5} | b{b_no:<2} | {text[:70]}")

# ───── 원본: TRPG_Rule · 셀 18 ─────
# Ex12. 숫자로 확인
# import random
#
# lens = sorted(len(c["text"]) for c in chunks)
# print("길이 min/중앙/max:", lens[0], lens[len(lens) // 2], lens[-1])
# print("빈 텍스트:", sum(not c["text"].strip() for c in chunks))
# print("탭 잔존:", sum("\t" in c["text"] for c in chunks))
# print("머리글 잔존:", sum(HEADER_TEXT in c["text"] for c in chunks))
# print("점선 잔존:", sum(bool(re.search(r"\.{4,}", c["text"])) for c in chunks))
#
# covered = set()
# for c in chunks:
#     covered.update(range(c["meta"]["page"], c["meta"]["page_end"] + 1))
# missing = [p for p in range(1, doc.page_count + 1) if p not in covered]
# print("청크가 닿지 않은 쪽:", len(missing), missing[:10])
#
# print("\n[장별 청크 수]")
# for h1, n in Counter(c["meta"]["h1"] for c in chunks).items():
#     print(f"{n:4} | {h1}")
#
# no_title = [w for w in wants if not any(w in c["meta"]["title"] for c in chunks)]
# print(f"\n골든 정답 제목 확보: {len(wants) - len(no_title)}/{len(wants)}", no_title)
#
# by_len = sorted(chunks, key=lambda c: len(c["text"]))
# print("\n[가장 짧은 8개]")
# for c in by_len[:8]:
#     print(len(c["text"]), c["meta"]["page"], repr(c["text"][:80]))
# print("\n[가장 긴 5개]")
# for c in by_len[-5:]:
#     print(len(c["text"]), c["meta"]["page"], c["meta"]["title"])
#
# random.seed(0)
# print("\n[무작위 5개]")
# for c in random.sample(chunks, 5):
#     print(c["id"], "p.", c["meta"]["page"])
#     print(c["text"][:250], "\n---")

# ───── 원본: TRPG_Rule · 셀 19 ─────
# Ex13. jsonl 저장 + 다시 읽어 비교
# with open(OUT_PATH, "w", encoding="utf-8") as f:
#     for c in chunks:
#         f.write(json.dumps(c, ensure_ascii=False) + "\n")   # 한 줄 = 청크 하나
#
# back = [json.loads(line) for line in open(OUT_PATH, encoding="utf-8")]
# print("저장:", len(chunks), "| 다시 읽음:", len(back), "| 동일:", back == chunks)

### 📦 데이터 준비 · 이상 문자 정리 (제어문자·소프트 하이픈·제로폭 공백) 후 바뀐 청크만 다시 임베딩
- 문제 청크 발견 → 정리 → 바뀐 청크 12개만 임베딩 다시 만들기, 남은 이상 문자 0
- 출처: `TRPG_Rule` 셀 23, `TRPG_Rule` 셀 24, `TRPG_Rule` 셀 25

In [ ]:
# ───── 원본: TRPG_Rule · 셀 23 ─────
# Ex33. 보이지 않는 제어문자 찾기
# import re, collections
# BAD = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")      # 엑셀이 거부하는 문자들
#
# cnt, bad_chunks = collections.Counter(), []
# for r in rows:
#     found = BAD.findall(r["text"])
#     if found:
#         bad_chunks.append(r)
#         cnt.update(repr(ch) for ch in found)
#
# print("문제 청크 수:", len(bad_chunks), "/", len(rows))
# print("문자 종류별 개수:", dict(cnt))
# for r in bad_chunks[:8]:
#     s = BAD.search(r["text"]).start()
#     print(r["meta"]["title"], "| p.", r["meta"]["page"], "|", repr(r["text"][max(0, s-30):s+30]))

# ───── 원본: TRPG_Rule · 셀 24 ─────
# Ex34. 이상 문자 점검 + 정리
# import unicodedata
#
# 1) 눈에 안 보이는 문자(제어/서식/사용자정의) 전수 점검
# cats = collections.Counter()
# for r in rows:
#     for ch in r["text"]:
#         if unicodedata.category(ch) in ("Cc", "Co", "Cf", "Cs") and ch != "\n":
#             cats[(hex(ord(ch)), unicodedata.category(ch))] += 1
# print("이상 문자:", dict(cats))
#
# 2) \x07 제거 후 jsonl 다시 저장
# for r in rows:
#     r["text"] = BAD.sub("", r["text"])
# with open(OUT_PATH, "w", encoding="utf-8") as f:
#     for r in rows:
#         f.write(json.dumps(r, ensure_ascii=False) + "\n")
#
# 3) 그 청크 1개만 임베딩 다시 만들기 (chunks, emb는 Ex14에서 만든 것)
# i = next(i for i, c in enumerate(chunks) if BAD.search(c["text"]))
# chunks[i]["text"] = BAD.sub("", chunks[i]["text"])
# emb[i] = np.array(embed_batch([chunks[i]["text"]])[0], dtype="float32")
# np.save(EMB_PATH, emb)
# print("수정한 청크:", i, chunks[i]["meta"]["title"])

# ───── 원본: TRPG_Rule · 셀 25 ─────
# Ex35. 발견된 문제 수정 + 바뀐 청크만 임베딩 다시 만들기
# def clean_text(s):
#     return re.sub(r"[\x07\xad\u200b]", "", s)        # 제어문자, 숨은 하이픈, 폭 없는 공백 제거
#
# FIX = {"rule_1754": ("Iron Golem", "Knight"),
#        "rule_0712": ("Castingin Armor", "Casting in Armor"),
#        "rule_1726": ("GrayOoze", "Gray Ooze")}
#
# def fix_chunk(c):
#     c["text"] = clean_text(c["text"])
#     if c["id"] in FIX:
#         old, new = FIX[c["id"]]
#         c["text"] = c["text"].replace(old, new)
#         if c["meta"]["title"] == old:
#             c["meta"]["title"] = new
#     return c
#
# 1) jsonl 전체 고쳐서 저장
# rows = [fix_chunk(r) for r in rows]
# with open(OUT_PATH, "w", encoding="utf-8") as f:
#     for r in rows:
#         f.write(json.dumps(r, ensure_ascii=False) + "\n")
#
# 2) 검색용 chunks도 고치고, 바뀐 것만 임베딩 다시 만들기
# changed = []
# for i, c in enumerate(chunks):
#     before = c["text"]
#     fix_chunk(c)
#     if c["text"] != before or c["id"] in FIX:
#         changed.append(i)
# print("바뀐 청크:", len(changed))
#
# new_vecs = np.array(embed_batch([chunks[i]["text"] for i in changed]), dtype="float32")
# emb[changed] = new_vecs
# np.save(EMB_PATH, emb)
#
# 3) 확인
# k = next(c for c in chunks if c["id"] == "rule_1754")
# print(k["meta"]["title"], "|", k["text"][:60].replace("\n", " / "))
# print("남은 이상 문자:", sum(len(re.findall(r"[\x07\xad\u200b]", c["text"])) for c in chunks))

### 🧪 단순 테스트 · 청킹 결과를 눈으로 보려고 텍스트/엑셀로 내보내기, GPU 확인
- 단순 확인용 코드. 본문 흐름에는 영향 없음
- 출처: `TRPG_Rule` 셀 2, `TRPG_Rule` 셀 21, `TRPG_Rule` 셀 22

In [ ]:
# ───── 원본: TRPG_Rule · 셀 2 ─────
# GPU 확인
# import torch
#
# print("GPU 사용 가능:", torch.cuda.is_available())
# if torch.cuda.is_available():
#     print("GPU 이름:", torch.cuda.get_device_name(0))

# ───── 원본: TRPG_Rule · 셀 21 ─────
# Ex31. 청킹 결과를 읽기 쉬운 파일로 내보내기
# import json, csv
# from google.colab import files
#
# rows = [json.loads(line) for line in open(OUT_PATH, encoding="utf-8")]   # rules_new.jsonl
# print("청크 수:", len(rows))
#
# 1) 읽기용 txt: 청크마다 구분선 + (번호, 쪽, 종류, 제목) + 본문
# TXT_PATH = f"{DATA}/rules_new_view.txt"
# with open(TXT_PATH, "w", encoding="utf-8") as f:
#     for r in rows:
#         m = r["meta"]
#         f.write(f"===== #{r['id']} | p.{m['page']} | {m['type']} | {m['title']} =====\n")
#         f.write(r["text"] + "\n\n")
#
# 2) 엑셀용 csv: 정렬·필터·검색용 (utf-8-sig라야 엑셀에서 한글이 안 깨져요)
# CSV_PATH = f"{DATA}/rules_new_view.csv"
# with open(CSV_PATH, "w", encoding="utf-8-sig", newline="") as f:
#     w = csv.writer(f)
#     w.writerow(["id", "type", "h1", "section", "title", "page", "글자수", "text"])
#     for r in rows:
#         m = r["meta"]
#         w.writerow([r["id"], m["type"], m["h1"], m["section"], m["title"], m["page"], len(r["text"]), r["text"]])
#
# print("저장:", TXT_PATH, "|", CSV_PATH)
# files.download(TXT_PATH)     # 내 컴퓨터로 받기
# files.download(CSV_PATH)

# ───── 원본: TRPG_Rule · 셀 22 ─────
# Ex32. 보기 좋은 엑셀 만들기 (openpyxl: Colab에 설치돼 있음)
# from openpyxl import Workbook
# from openpyxl.styles import Font, PatternFill, Alignment
# from openpyxl.utils import get_column_letter
#
# wb = Workbook()
# ws = wb.active
# ws.title = "청크"
#
# head = ["id", "쪽", "종류", "장(h1)", "제목", "글자수", "미리보기(앞 150자)", "전체 본문(클릭하면 위 입력줄에 표시)"]
# ws.append(head)
#
# for r in rows:
#     m = r["meta"]
#     preview = r["text"][:150].replace("\n", " ")        # 줄바꿈 → 공백
#     ws.append([r["id"], m["page"], m["type"], m["h1"], m["title"], len(r["text"]), preview, r["text"]])
#
# 열 너비
# for col, width in zip("ABCDEFGH", [7, 6, 7, 18, 30, 8, 90, 40]):
#     ws.column_dimensions[col].width = width
#
# 머리글 꾸미기 + 고정 + 필터
# for c in ws[1]:
#     c.font = Font(bold=True, color="FFFFFF")
#     c.fill = PatternFill("solid", fgColor="305496")
#     c.alignment = Alignment(vertical="center")
# ws.freeze_panes = "A2"
# ws.auto_filter.ref = ws.dimensions
#
# 본문 줄바꿈 끄고 한 줄 높이로 고정 (줄이 길어져도 행 높이가 안 늘어나게)
# for row in ws.iter_rows(min_row=2):
#     for c in row:
#         c.alignment = Alignment(vertical="top", wrap_text=False)
#     ws.row_dimensions[row[0].row].height = 16
#
# XLSX_PATH = f"{DATA}/rules_new_view.xlsx"
# wb.save(XLSX_PATH)
# print("저장:", XLSX_PATH)
# files.download(XLSX_PATH)

### 🔍 진단 · 첫 검색 점검: 영어 질문에서 못 찾은 3개(Prone, Blinded, Hide) 원인 보기
- 영어 25문항 1등 22/25. 못 찾은 것: Prone / Blinded / Hide
- Hide는 정답 청크 점수(0.482)가 1등(0.558)보다 낮았음 → 이후에도 알려진 약점
- 출처: `TRPG_Rule` 셀 27, `TRPG_Rule` 셀 28, `TRPG_Rule` 셀 29

In [ ]:
# ───── 원본: TRPG_Rule · 셀 27 ─────
# Ex16. 검색 확인
# golden = json.load(open(f"{DATA}/golden.json", encoding="utf-8"))
# ALIAS = {"Grappl": {"grappled", "grappling"}}       # 줄여 적은 정답 보정
#
# def norm(title):
#     title = re.sub(r"\s*\[[^\]]*\]$", "", title)      # 이름 끝의 [..] 표시 제거
#     title = re.sub(r"\s*\([^)]*\)$", "", title)       # 이름 끝의 (..) 표시 제거
#     return title.strip().lower()
#
# def is_hit(title, want):                                # # 이름이 정확히 같을 때만 정답
#     return norm(title) == norm(want) or norm(title) in ALIAS.get(want, ())
#
# def wants_of(g):
#     return [g["want"]] if isinstance(g["want"], str) else g ["want"]
#
# K = 3
# q_vecs = np.array(embed_batch([g["q"] for g in golden]), dtype="float32")
# faiss.normalize_L2(q_vecs)
# scores, ids = index.search(q_vecs, K)       # ids[i] = 질문 i와 가까운 조각 번호 3개
#
# top1 = top3 = 0
# for g, row in zip(golden, ids):
#     titles = [chunks[i]["meta"]["title"] for i in row]
#     hits = [any(is_hit(t, w) for w in wants_of(g) for t in titles)] # 3개 각각 정답인지
#     top1 += hits[0]                                                 # True는 1로 더해짐
#     top3 += any(hits)
#     if not hits[0]:                                         # 1등이 정답이 아닌 질문만 출력
#         mark = "3개 안에 있음" if any(hits) else "못 찾음"
#         print(f"[{mark}] {g['q']})\n       정답: {g['want']} | 검색: {titles}")
#
# print(f"1등이 정답: {top1}/{len(golden)} | 3개 안에 정답: {top3}/{len(golden)}")

# ───── 원본: TRPG_Rule · 셀 28 ─────
# Ex17. 못 찾은 3개: 정답이 상위 30개 중 몇 등에 있나
# miss = [g for g in golden if g["want"] in ("Prone", "Blinded", "Hide")]
# q = np.array(embed_batch([g["q"] for g in miss]), dtype="float32")
# faiss.normalize_L2(q)
# scores, ids = index.search(q, 30)
#
# for g, srow, row in zip(miss, scores, ids):
#     rank = next((r + 1 for r, i in enumerate(row)
#                  if any(is_hit(chunks[i]["meta"]["title"], w) for w in wants_of(g))), None)
#     print(g["q"], "->", "정답 등수:", rank)
#     if rank:
#         print("   1등 점수:", round(float(srow[0]), 3), "| 정답 점수:", round(float(srow[rank - 1]), 3))
#         print("   정답 조각:", chunks[row[rank - 1]]["text"][:150].replace("\n", " | "))

# ───── 원본: TRPG_Rule · 셀 29 ─────
# Ex18. 정답 조각의 실제 내용과 질문과의 점수
# for g, qv, srow in zip(miss, q, scores):
#     if g["want"] == "Blinded":
#         continue
#     print("\n질문:", g["q"], "| 1등 점수:", round(float(srow[0]), 3))
#     for i, c in enumerate(chunks):
#         if any(is_hit(c["meta"]["title"], w) for w in wants_of(g)):
#             print(f"  정답 후보 [{i}] p.{c['meta']['page']} 길이 {len(c['text'])} | 점수 {float(vecs[i] @ qv):.3f}")
#             print("   ", c["text"][:300].replace("\n", " | "))

### ✅ 채택·반영 · 첫 RAG 답변과 프롬프트 교체 (원문 먼저 복사 → 한국어 설명 / 예시 1개 추가)
- 수업 [4] Ex06·Ex11 방식을 적용. '원문 먼저 복사 + 예시 1개' 구조는 현재 `RAG_TEMPLATE`(Ex09-c)에 그대로 이어짐
- 첫 버전은 임베딩만 쓰는 `search_rules`와 Colab 변수(`LLM_MODEL`)를 사용
- 출처: `TRPG_Rule` 셀 30, `TRPG_Rule` 셀 31, `TRPG_Rule` 셀 32, `TRPG_Rule` 셀 33

In [ ]:
# ───── 원본: TRPG_Rule · 셀 30 ─────
# Ex19. 검색 → 근거 만들기 → 프롬프트 채우기 → 답변 (+ 중국어 섞이면 다시 시도)
# def search_rules(question, k=5):
#     q = np.array(embed_batch([question]), dtype="float32")
#     faiss.normalize_L2(q)
#     scores, ids = index.search(q, k)
#     return [chunks[i] for i in ids[0]]                 # 비슷한 룰북 조각 k개
#
# SYSTEM = ("You are a D&D rules assistant. Always answer in Korean (Hangul) only. "
#           "Never use Chinese characters or Chinese words.")
#
# def ask_llm(prompt, model=LLM_MODEL, temperature=0.1):
#     res = requests.post(
#         f"{OLLAMA_URL}/api/chat",
#         json={"model": model,
#               "messages": [{"role": "system", "content": SYSTEM},     # 수업 [3]A Ex13: system 역할
#                            {"role": "user", "content": prompt}],
#               "stream": False,
#               "options": {"num_ctx": 8192, "temperature": temperature}},
#         timeout=300,
#     )
#     res.raise_for_status()
#     return res.json()["message"]["content"]
#
# TERMS = ("Blinded=실명, Advantage=이점, Disadvantage=불리점, Ability Check=능력 판정, "
#          "Attack Roll=공격 굴림, Saving Throw=내성 굴림")      # 용어 통일 (필요하면 계속 추가)
#
# RAG_TEMPLATE = """당신은 D&D 규칙 안내자입니다. 아래 규칙을 반드시 지키세요.
# 1. [근거]에 적힌 내용만 사용하세요. 근거에 없으면 "근거에서 찾지 못했어요"라고 답하세요.
# 2. 질문과 관련 없는 근거는 무시하세요.
# 3. 한국어로만 답하세요. 규칙 용어는 영어 원문을 괄호로 함께 적으세요. 예: 실명(Blinded)
# 4. 용어는 아래 표의 한국어만 쓰고, 음역(예: 블라인드)은 쓰지 마세요: {terms}
# 5. 근거에 있는 효과를 빠짐없이 쓰되 다섯 문장 이내로 쓰세요. 이점과 불리점은 누구의 굴림인지(자신의 공격인지, 자신을 향한 공격인지) 구분해서 쓰세요.
#
### 근거
# {context}
#
### 질문
# {question}
#
### 답변 (한국어로만)"""
#
# def has_chinese(text):                                 # 중국어 글자가 섞였는지 코드로 확인
#     return bool(re.search(r"[\u4e00-\u9fff]", text))
#
# def rag_answer(question, k=5, tries=3):
#     found = search_rules(question, k)
#     lines = []
#     for c in found:                                    # "[제목 p.쪽] 본문"
#         lines.append(f"[{c['meta']['title']} p.{c['meta']['page']}] {c['text'][:2000]}")
#     context = "\n\n".join(lines)
#     prompt = RAG_TEMPLATE.format(terms=TERMS, context=context, question=question)
#
#     for n in range(1, tries + 1):                      # 중국어가 섞이면 temperature를 올려 다시 생성
#         answer = ask_llm(prompt, temperature=0.1 + 0.3 * (n - 1))
#         if not has_chinese(answer):
#             break
#     sources = [f"{c['meta']['title']} (p.{c['meta']['page']})" for c in found]
#     return {"answer": answer, "sources": sources, "tries": n, "ok": not has_chinese(answer)}
#
# for g in golden[:6]:                                   # 골든 질문 앞 6개
#     r = rag_answer(g["q"])
#     found = any(is_hit(s.rsplit(" (p.", 1)[0], w) for s in r["sources"] for w in wants_of(g))
#     print("질문:", g["q"])
#     print("답변:", r["answer"])
#     print("시도:", r["tries"], "| 정답 항목이 출처에 있음:", found, "\n")

# ───── 원본: TRPG_Rule · 셀 31 ─────
# Ex20. 틀린 답변의 근거 조각 확인 (기회 공격 / 활 쏘기 / 집중)
# for idx in (1, 4, 5):
#     g = golden[idx]
#     print("질문:", g["q"])
#     for c in search_rules(g["q"]):
#         if any(is_hit(c["meta"]["title"], w) for w in wants_of(g)):
#             print(c["text"][:900], "\n")

# ───── 원본: TRPG_Rule · 셀 32 ─────
# Ex21. 프롬프트 교체: 근거 문장 먼저 복사 → 한국어 설명  (수업 [4] Ex06, Ex11)
# TERMS = ("Blinded=실명, Advantage=이점, Disadvantage=불리점, Ability Check=능력 판정, "
#          "Attack Roll=공격 굴림, Saving Throw=내성 굴림, Opportunity Attack=기회 공격, "
#          "Reaction=반응, Disengage=이탈, Concentration=집중, Incapacitated=행동 불능")
#
# RAG_TEMPLATE = """당신은 D&D 규칙 안내자입니다. 아래 규칙과 형식을 반드시 지키세요.
#
# 규칙
# 1. [근거]에 적힌 내용만 사용하세요. 근거에 없으면 "근거에서 찾지 못했어요"라고만 답하세요.
# 2. 질문과 관련 없는 근거는 무시하세요.
# 3. 설명은 한국어로만 쓰고, 규칙 용어는 영어 원문을 괄호로 함께 적으세요. 예: 실명(Blinded)
# 4. 용어는 아래 표의 한국어만 쓰고, 음역은 쓰지 마세요: {terms}
# 5. 근거의 내용을 빼거나 뒤집지 마세요. 특히 Advantage와 Disadvantage를 바꾸지 마세요.
#
# 형식
# [근거 문장] 질문에 답하는 핵심 문장을 근거에서 영어 그대로 한두 문장 복사
# [설명] 위 문장을 바탕으로 한 한국어 설명 (다섯 문장 이내)
#
### 근거
# {context}
#
### 질문
# {question}
#
### 답변"""
#
# 답에 반드시 들어가야 할 말 (골든 질문 번호: 단어). 수업 [4] Ex04처럼 코드로 확인
# MUST = {0: ["+2", "+5"], 1: ["Reaction"], 3: ["20"], 4: ["Disadvantage"], 5: ["Constitution"]}
#
# passed = 0
# for idx, words in MUST.items():
#     r = rag_answer(golden[idx]["q"])
#     ok = all(w in r["answer"] for w in words)
#     passed += ok
#     print("통과" if ok else "실패", "|", golden[idx]["q"])
#     print(r["answer"], "\n")
# print(f"통과 {passed}/{len(MUST)}")

# ───── 원본: TRPG_Rule · 셀 33 ─────
# Ex22. 프롬프트 교체: 예시 1개 추가 (수업 [4] Ex06 In-Context Learning)
# RAG_TEMPLATE = """당신은 D&D 규칙 안내자입니다. 아래 규칙을 반드시 지키세요.
# 1. [근거]에 적힌 내용만 사용하세요. 근거에 없으면 "근거에서 찾지 못했어요"라고만 답하세요.
# 2. 질문과 관련 없는 근거는 무시하세요.
# 3. 용어는 이 표의 한국어를 쓰고 영어 원문을 괄호로 함께 적으세요: {terms}
# 4. 근거의 조건, 숫자, 이점/불리점을 빼거나 바꾸지 마세요.
# 5. 답변은 "원문:"과 "설명:" 두 줄로 쓰세요. 원문은 질문에 직접 답하는 문장을 근거에서 영어 그대로 복사하고, 설명은 한국어로 쓰세요.
#
### 예시
# 질문: What does the Dash action do?
# 근거: [Dash [Action] p.181] When you take the Dash action, you gain extra movement for the current turn. The increase equals your Speed after applying any modifiers.
# 답변:
# 원문: When you take the Dash action, you gain extra movement for the current turn. The increase equals your Speed after applying any modifiers.
# 설명: 질주(Dash) 행동을 하면 이번 턴에 속도(Speed)만큼 이동 거리가 늘어납니다.
#
### 근거
# {context}
#
### 질문
# {question}
#
### 답변"""
#
# passed = 0
# for idx, words in MUST.items():
#     r = rag_answer(golden[idx]["q"])
#     ok = all(w in r["answer"] for w in words)
#     passed += ok
#     print("통과" if ok else "실패", "|", golden[idx]["q"])
#     print(r["answer"], "\n")
# print(f"통과 {passed}/{len(MUST)}")

### 📦 데이터 준비 · 한국어 질문 시험지 28문항(golden_ko) 만들기
- 이후 `golden_ko.json`으로 저장돼 본문에서 불러 씀. 처음엔 '홀드아웃'이었지만 이후 검색·프롬프트를 손보면서 오염됨
- 출처: `TRPG_Rule` 셀 34

In [ ]:
# ───── 원본: TRPG_Rule · 셀 34 ─────
# Ex23. 한국어 질문 시험지 (홀드아웃: 이 질문들을 보고 검색·프롬프트를 손보지 않는다)
# golden_ko = [
#     {"q": "독에 중독되면 뭐가 불리해져?", "want": "Poisoned"},
#     {"q": "몸이 마비되면 공격받을 때 어떻게 돼?", "want": "Paralyzed"},
#     {"q": "겁에 질린 상태에서는 적한테 다가갈 수 있어?", "want": "Frightened"},
#     # {"q": "기절하면 움직이거나 말할 수 있어?", "want": "Stunned"},
#     # {"q": "의식을 잃으면 들고 있던 걸 떨어뜨려?", "want": "Unconscious"},
#     # {"q": "밧줄에 묶여서 움직일 수 없으면 무슨 상태야?", "want": "Restrained"},
#     # {"q": "매료된 상태에서 나를 홀린 상대를 공격해도 돼?", "want": "Charmed"},
#     # {"q": "행동 불능이 되면 뭘 못 해?", "want": "Incapacitated"},
#     # {"q": "긴 휴식은 몇 시간 쉬어야 해?", "want": "Long Rest"},
#     # {"q": "전투 시작할 때 누가 먼저 움직일지 어떻게 정해?", "want": "Initiative"},
#     # {"q": "이번 턴에 더 멀리 이동하려면 무슨 행동을 해?", "want": "Dash"},
#     # {"q": "적한테 기회 공격 안 맞고 도망가려면?", "want": "Disengage"},
#     # {"q": "상대가 뭔가 하면 그때 반응해서 행동하려면 어떻게 해?", "want": "Ready"},
#     # {"q": "숨겨진 걸 찾으려면 어떤 행동을 해야 해?", "want": "Search"},
#     # {"q": "상대를 설득하거나 협박하려면 무슨 행동이야?", "want": "Influence"},
#     # {"q": "주문을 쓰는 행동은 뭐라고 불러?", "want": "Magic"},
#     # {"q": "물건을 쓰는 데도 행동이 필요해?", "want": "Utilize"},
#     # {"q": "물속에서 싸우면 무기에 제한이 있어?", "want": "Underwater Combat"},
#     # {"q": "말을 타고 싸울 때 규칙이 있어?", "want": "Mounted Combat"},
#     # {"q": "숨을 못 쉬게 되면 얼마나 버틸 수 있어?", "want": "Suffocating"},
#     # {"q": "안개가 짙은 곳에서는 시야가 어떻게 돼?", "want": "Heavily Obscured"},
#     # {"q": "어슴푸레한 숲에서는 보이는 게 어때?", "want": "Lightly Obscured"},
#     # {"q": "피해에 저항이 있으면 얼마나 줄어?", "want": "Resistance"},
#     # {"q": "약점이 있는 몬스터는 피해를 얼마나 더 받아?", "want": "Vulnerability"},
#     # {"q": "화살통에는 화살이 몇 개 들어가?", "want": "Quiver"},
#     # {"q": "작은 유리병에는 몇 온스가 들어가?", "want": "Vial"},
#     # {"q": "무작위 효과가 나오는 마법 지팡이가 있어?", "want": "Wand of Wonder"},
#     # {"q": "탈것으로 쓰는 말은 얼마야?", "want": "Riding Horse"},
#     {"q": "숙련을 두 배로 쓰는 건 뭐야?", "want": "Expertise"},
# ]
#
# 정답 이름이 룰북에 실제로 있는지 확인 (is_hit은 Ex16에서 정의)
# bad = [g["want"] for g in golden_ko
#        if not any(is_hit(c["meta"]["title"], g["want"]) for c in chunks)]
# print("질문 수:", len(golden_ko), "| 룰북에 없는 정답 이름:", bad)
#
# golden_ko = [g for g in golden_ko if g["want"] not in bad]       # 없는 건 시험지에서 뺌
# json.dump(golden_ko, open(f"{DATA}/golden_ko.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
# print("저장한 질문 수:", len(golden_ko))

### ✅ 채택·반영 · 한국어 질문 → 영어 검색어 바꿔 쓰기 첫 시도 + 용어표 적용
- 용어표 적용 결과: 한국어 (1등/3개/5개) = (10, 19, 21)/28, 영어 = (21, 22, 23)/25
- 여기서 출발해 번역 ①②(Ex05-b)로 발전
- 출처: `TRPG_Rule` 셀 35, `TRPG_Rule` 셀 36, `TRPG_Rule` 셀 37, `TRPG_Rule` 셀 38

In [ ]:
# ───── 원본: TRPG_Rule · 셀 35 ─────
# Ex24. 한국어 질문 검색 확인 (챗봇이 근거를 읽는 개수인 5개까지 봄)
# K = 5
# q_vecs = np.array(embed_batch([g["q"] for g in golden_ko]), dtype="float32")
# faiss.normalize_L2(q_vecs)
# scores, ids = index.search(q_vecs, K)
#
# top1 = top3 = top5 = 0
# for g, row in zip(golden_ko, ids):
#     titles = [chunks[i]["meta"]["title"] for i in row]
#     hits = [is_hit(t, g["want"]) for t in titles]
#     top1 += hits[0]
#     top3 += any(hits[:3])
#     top5 += any(hits)
#     if not hits[0]:
#         mark = "5개 안에 있음" if any(hits) else "못 찾음"
#         print(f"[{mark}] {g['q']}\n      정답: {g['want']} | 검색: {titles[:3]}")
#
# n = len(golden_ko)
# print(f"1등이 정답: {top1}/{n} | 3개 안: {top3}/{n} | 5개 안: {top5}/{n}")

# ───── 원본: TRPG_Rule · 셀 36 ─────
# Ex25. 질문 바꿔 쓰기 : 한국어 질문 -> 영어 검색어
# def ask_plain(prompt, temperature=0.0):                # 한국어 고정 system 없이 호출
#     res = requests.post(
#         f"{OLLAMA_URL}/api/chat",
#         json={"model": LLM_MODEL,
#               "messages": [{"role": "user", "content": prompt}],
#               "stream": False,
#               "options": {"temperature": temperature}},
#         timeout=300,
#     )
#     res.raise_for_status()
#     return res.json()["message"]["content"]
#
# REWRITE_TEMPLATE = """Convert the user's question about D&D 5.2 rules into a short English search query.
# Use the official rule terms (conditions, actions, and so on) that the rulebook would use.
# Output only the query, nothing else.
#
# Question: 숙련 보너스는 레벨마다 얼마야?
# Query: Proficiency Bonus by character level
#
# Question: 레벨업하려면 경험치가 얼마나 필요해?
# Query: Experience Points needed to advance level
#
# Question: 방어도는 어떻게 계산해?
# Query: Armor Class calculation
#
# Question: {question}
# Query:"""
#
# def rewrite_query(question):
#     return ask_plain(REWRITE_TEMPLATE.format(question=question)).strip().split("\n")[0]
#
# def eval_queries(items, queries, K=5):                 # 검색어 목록으로 1등/3개/5개 안 정답 수 세기
#     q_vecs = np.array(embed_batch(queries), dtype="float32")
#     faiss.normalize_L2(q_vecs)
#     scores, ids = index.search(q_vecs, K)
#     t1 = t3 = t5 = 0
#     for g, row in zip(items, ids):
#         hits = [any(is_hit(chunks[i]["meta"]["title"], w) for w in wants_of({"want": g["want"]})) for i in row]
#         t1 += hits[0]; t3 += any(hits[:3]); t5 += any(hits)
#     return t1, t3, t5
#
# 1) 한국어 시험지 28개: 원래 질문 vs 바꿔 쓴 검색어
# ko_q = [g["q"] for g in golden_ko]
# ko_en = [rewrite_query(q) for q in ko_q]
# for q, e in zip(ko_q, ko_en):
#     print(q, "->", e)
# print("\n한국어 | 원래 질문   (1등/3개/5개):", eval_queries(golden_ko, ko_q), "/", len(golden_ko))
# print("한국어 | 바꿔 쓴 검색어 (1등/3개/5개):", eval_queries(golden_ko, ko_en), "/", len(golden_ko))
#
# 2) 영어 골든 25개: 바꿔 써도 나빠지지 않는지 확인
# en_q = [g["q"] for g in golden]
# en_en = [rewrite_query(q) for q in en_q]
# print("\n영어   | 원래 질문   (1등/3개/5개):", eval_queries(golden, en_q), "/", len(golden))
# print("영어   | 바꿔 쓴 검색어 (1등/3개/5개):", eval_queries(golden, en_en), "/", len(golden))

# ───── 원본: TRPG_Rule · 셀 37 ─────
# Ex26. 숫자가 왜 다른지 확인 (같은 셀 안에서 두 방식을 비교)
# q_vecs = np.array(embed_batch([g["q"] for g in golden]), dtype="float32")
# faiss.normalize_L2(q_vecs)
# scores, ids = index.search(q_vecs, 5)
#
# top1 = 0
# for g, row in zip(golden, ids):
#     titles = [chunks[i]["meta"]["title"] for i in row]
#     hits = [any(is_hit(t, w) for w in wants_of(g)) for t in titles]
#     top1 += hits[0]
#     if any(hits) and not hits[0]:
#         print("1등 아님:", g["q"], "| 정답:", g["want"], "| 검색:", titles)
#
# print("직접 센 1등:", top1)
# print("eval_queries:", eval_queries(golden, [g["q"] for g in golden]))

# ───── 원본: TRPG_Rule · 셀 38 ─────
# Ex27. 용어표: 검색어 바꿔 쓰기 + 답변에 같이 사용
# TERM_MAP = {                                           # 한국어 = 룰북 영어 용어
#     # 상태
#     "실명": "Blinded", "매료": "Charmed", "귀먹음": "Deafened", "탈진": "Exhaustion",
#     "공포": "Frightened", "붙잡힘": "Grappled", "행동 불능": "Incapacitated", "투명": "Invisible",
#     "마비": "Paralyzed", "석화": "Petrified", "중독": "Poisoned", "넘어짐": "Prone",
#     "포박": "Restrained", "기절": "Stunned", "의식 불명": "Unconscious",
#     # 행동
#     "공격": "Attack", "질주": "Dash", "이탈": "Disengage", "회피": "Dodge", "도움": "Help",
#     "숨기": "Hide", "영향": "Influence", "마법": "Magic", "준비": "Ready", "탐색": "Search",
#     "연구": "Study", "사용": "Utilize",
#     # 판정·전투
#     "이점": "Advantage", "불리점": "Disadvantage", "능력 판정": "Ability Check",
#     "공격 굴림": "Attack Roll", "내성 굴림": "Saving Throw", "기회 공격": "Opportunity Attack",
#     "반응": "Reaction", "추가 행동": "Bonus Action", "집중": "Concentration",
#     "이니셔티브": "Initiative", "저항": "Resistance", "취약": "Vulnerability",
#     "엄폐": "Cover", "치명타": "Critical Hit", "숙련": "Proficiency", "전문화": "Expertise",
#     "긴 휴식": "Long Rest", "짧은 휴식": "Short Rest",
#     # 능력치
#     "근력": "Strength", "민첩": "Dexterity", "건강": "Constitution",
#     "지능": "Intelligence", "지혜": "Wisdom", "매력": "Charisma",
# }
# TERMS = ", ".join(f"{en}={ko}" for ko, en in TERM_MAP.items())     # 답변용 (영어=한국어)
# HINTS = ", ".join(f"{ko}={en}" for ko, en in TERM_MAP.items())     # 검색어용 (한국어=영어)
#
# REWRITE_TEMPLATE = """Convert the user's question about D&D 5.2 rules into a short English search query.
# Use the official rule terms that the rulebook would use. Korean-to-English term hints: {hints}
# Output only the query, nothing else.
#
# Question: 숙련 보너스는 레벨마다 얼마야?
# Query: Proficiency Bonus by character level
#
# Question: 레벨업하려면 경험치가 얼마나 필요해?
# Query: Experience Points needed to advance level
#
# Question: 방어도는 어떻게 계산해?
# Query: Armor Class calculation
#
# Question: {question}
# Query:"""
#
# def rewrite_query(question):
#     out = ask_plain(REWRITE_TEMPLATE.format(hints=HINTS, question=question))
#     return out.strip().split("\n")[0].replace("Query:", "").strip()
#
# ko_en = [rewrite_query(q) for q in ko_q]
# en_en = [rewrite_query(q) for q in en_q]
# print("한국어 | 용어표 적용 (1등/3개/5개):", eval_queries(golden_ko, ko_en), "/", len(golden_ko))
# print("영어   | 용어표 적용 (1등/3개/5개):", eval_queries(golden, en_en), "/", len(golden))

### ✅ 채택·반영 · 근거 번호를 먼저 고르게 하기 + 정의 조각 우선 규칙
- Ex29(번호 먼저 고르기): 이후 2단계 선택(`select_idx`)으로 발전 — 채택
- Ex30(정의 조각 우선 규칙): 저장된 출력이 없고 이후 노트북에 이어지지 않음 — 채택 안 함
- 출처: `TRPG_Rule` 셀 40, `TRPG_Rule` 셀 41

In [ ]:
# ───── 원본: TRPG_Rule · 셀 40 ─────
# Ex29. 근거 번호를 먼저 고르게 하기 (수업 [4] Ex06 예시 + Ex11 CoT)
# RAG_TEMPLATE = """당신은 D&D 규칙 안내자입니다. 아래 규칙을 반드시 지키세요.
# 1. [근거]는 번호가 붙은 여러 개입니다. 질문에 가장 직접 답하는 근거 하나를 고르세요. 관련 없는 근거는 무시하세요.
# 2. 고른 근거에 적힌 내용만 사용하세요. 어느 근거에도 없으면 "근거에서 찾지 못했어요"라고만 답하세요.
# 3. 용어는 이 표의 한국어를 쓰고 영어 원문을 괄호로 함께 적으세요: {terms}
# 4. 근거의 조건, 숫자, 이점/불리점을 빼거나 바꾸지 마세요.
# 5. 답변은 "근거 번호:", "원문:", "설명:" 세 줄로 쓰세요. 원문은 고른 근거에서 질문에 직접 답하는 문장을 영어 그대로 복사하고, 설명은 한국어로 쓰세요.
#
### 예시
# 질문: What does the Dash action do?
# 근거:
# [1] (Torch (1 CP) p.100) A Torch burns for 1 hour, casting Bright Light in a 20-foot radius and Dim Light for an additional 20 feet.
# [2] (Dash [Action] p.181) When you take the Dash action, you gain extra movement for the current turn. The increase equals your Speed after applying any modifiers.
# 답변:
# 근거 번호: 2
# 원문: When you take the Dash action, you gain extra movement for the current turn. The increase equals your Speed after applying any modifiers.
# 설명: 질주(Dash) 행동을 하면 이번 턴에 속도(Speed)만큼 이동 거리가 늘어납니다.
#
### 근거
# {context}
#
### 질문
# {question}
#
### 답변"""
#
# def rag_answer(question, k=5, tries=3):
#     query = rewrite_query(question)                    # 검색은 영어 검색어로
#     found = search_rules(query, k)
#     lines = []
#     for n, c in enumerate(found, 1):                   # "[번호] (제목 p.쪽) 본문"
#         lines.append(f"[{n}] ({c['meta']['title']} p.{c['meta']['page']}) {c['text'][:2000]}")
#     context = "\n".join(lines)
#     prompt = RAG_TEMPLATE.format(terms=TERMS, context=context, question=question)
#
#     for t in range(1, tries + 1):                      # 중국어가 섞이면 temperature를 올려 다시 생성
#         answer = ask_llm(prompt, temperature=0.1 + 0.3 * (t - 1))
#         if not has_chinese(answer):
#             break
#     sources = [f"{c['meta']['title']} (p.{c['meta']['page']})" for c in found]
#     return {"answer": answer, "sources": sources, "query": query, "tries": t}
#
# 1) 방금 틀린 질문 3개
# for q in ["넘어졌을 때 일어나려면 이동력이 얼마나 들어?",
#           "적한테서 숨으려면 무슨 판정을 해?",
#           "겁에 질리면 뭘 못 해?"]:
#     r = rag_answer(q)
#     print("질문:", q)
#     print("답변:", r["answer"])
#     print("출처:", r["sources"], "\n")
#
# 2) 이전에 통과했던 영어 5문항이 그대로 통과하는지 (MUST는 Ex21에서 정의)
# passed = 0
# for idx, words in MUST.items():
#     r = rag_answer(golden[idx]["q"])
#     ok = all(w in r["answer"] for w in words)
#     passed += ok
#     print("통과" if ok else "실패", "|", golden[idx]["q"])
#     if not ok:
#         print(r["answer"])
# print(f"통과 {passed}/{len(MUST)}")

# ───── 원본: TRPG_Rule · 셀 41 ─────
# Ex30. 정의 조각 우선 + 용어표 보강
# 1) TERM_MAP에 Stealth → 은신 항목을 같은 형식으로 추가하고, TERMS를 만드는 셀을 다시 실행
#
# 2) RAG_TEMPLATE의 1번 규칙만 아래로 바꾸기 (나머지는 Ex29 그대로)
# OLD_RULE1 = "1. [근거]는 번호가 붙은 여러 개입니다. 질문에 가장 직접 답하는 근거 하나를 고르세요. 관련 없는 근거는 무시하세요."
# NEW_RULE1 = ("1. [근거]는 번호가 붙은 여러 개입니다. 질문에 가장 직접 답하는 근거 하나를 고르세요. "
#              "제목에 [Condition], [Action]이 붙은 근거는 그 용어의 공식 정의이니, 질문이 그 용어를 묻는다면 우선하세요. "
#              "관련 없는 근거는 무시하세요.")
# assert OLD_RULE1 in RAG_TEMPLATE      # 못 찾으면 에러로 알려 줌
# RAG_TEMPLATE = RAG_TEMPLATE.replace(OLD_RULE1, NEW_RULE1)

### ❌ 기각 · 오답 원인 분석표 / 한국어 + 번역 두 번 검색해서 합치기 / 용어 27개 선택지 번역
- 두 번 검색 합치기: 합친 결과 8/17/19 vs 당시 8/18/20 → 이득 없음, 기각
- 용어 선택지 번역(Ex38)은 번역 ②(`REWRITE2`)로 채택됨
- 출처: `TRPG_Rule` 셀 42, `TRPG_Rule` 셀 43, `TRPG_Rule` 셀 44

In [ ]:
# ───── 원본: TRPG_Rule · 셀 42 ─────
# Ex36. 한국어 오답 원인 분석표 (golden_ko의 q, want 사용)
# def rank_of(text, g):
#     v = np.array(embed_batch([text]), dtype="float32"); faiss.normalize_L2(v)
#     sc, ids = index.search(v, 5)
#     titles = [chunks[i]["meta"]["title"] for i in ids[0]]
#     rank = next((r + 1 for r, t in enumerate(titles) if any(is_hit(t, w) for w in wants_of(g))), None)
#     return rank, titles
#
# out = []
# for g in golden_ko:
#     en = rewrite_query(g["q"])
#     r_raw, _ = rank_of(g["q"], g)        # 한국어 그대로
#     r_en, titles = rank_of(en, g)        # 영어로 바꾼 뒤
#     out.append((g["q"], en, g["want"], r_raw, r_en, titles))
#
# print("그대로 1등:", sum(o[3] == 1 for o in out), "| 바꾼 뒤 1등:", sum(o[4] == 1 for o in out))
# for q, en, want, r_raw, r_en, titles in out:
#     if r_en != 1:                         # 바꾼 뒤에도 1등이 아닌 것만
#         print(f"\n질문: {q}")
#         print(f"  영어로 바꾼 것: {en}")
#         print(f"  정답: {want} | 그대로 순위: {r_raw} | 바꾼 뒤 순위: {r_en}")
#         print(f"  1~3등: {titles[:3]}")

# ───── 원본: TRPG_Rule · 셀 43 ─────
# Ex37. 한국어 그대로 + 영어로 바꾼 것, 두 번 검색해서 합치기
# def search_both(q, k=5):
#     v = np.array(embed_batch([q, rewrite_query(q)]), dtype="float32")
#     faiss.normalize_L2(v)
#     sc, ids = index.search(v, k)
#     best = {}                                        # 청크 번호 → 두 검색 중 높은 점수
#     for srow, irow in zip(sc, ids):
#         for s, i in zip(srow, irow):
#             best[i] = max(best.get(i, -1.0), float(s))
#     top = sorted(best, key=best.get, reverse=True)[:k]
#     return [chunks[i] for i in top]
#
# t1 = t3 = t5 = 0
# for g in golden_ko:
#     titles = [c["meta"]["title"] for c in search_both(g["q"])]
#     hits = [any(is_hit(t, w) for w in wants_of(g)) for t in titles]
#     t1 += hits[0]; t3 += any(hits[:3]); t5 += any(hits)
# print(f"합친 검색 1등/3등/5등: {t1}/{t3}/{t5} (28문항)  | 지금: 8/18/20")

# ───── 원본: TRPG_Rule · 셀 44 ─────
# Ex38. 영어 용어 27개를 선택지로 주고 바꾸게 하기 (한국어 열쇠 단어 없음)
# RULE_TERMS = ["Blinded", "Charmed", "Deafened", "Exhaustion", "Frightened", "Grappled", "Incapacitated",
#               "Invisible", "Paralyzed", "Petrified", "Poisoned", "Prone", "Restrained", "Stunned", "Unconscious",
#               "Attack action", "Dash action", "Disengage action", "Dodge action", "Help action", "Hide action",
#               "Influence action", "Magic action", "Ready action", "Search action", "Study action", "Utilize action"]
#
# REWRITE2 = """Translate the user's D&D question into a short English search query for the D&D rules.
# If the question is about one of these official rule terms, use that term exactly as written.
# Official terms: {terms}
# Output only the query, nothing else.
#
# Question: {q}
# Query:"""
#
# def rewrite_query2(q):
#     return ask_plain(REWRITE2.format(terms=", ".join(RULE_TERMS), q=q)).strip()
#
# Ex37의 search_both에서 rewrite_query → rewrite_query2로 바꿔 한 번 더 측정

### ✅ 채택·반영 · 번역 방식별 비교표(A~E)와 최종 RRF
- 합계(48): A 한국어 그대로 (14,26,28) / B 예전 번역 (19,32,38) / C 용어 선택지 (25,30,32) / D 선택지+한국어 (25,30,36) / E 세 검색 RRF (26,38,41)
- E(세 갈래 RRF) 채택. 영어 회귀 (21,24,24)
- 출처: `TRPG_Rule` 셀 52, `TRPG_Rule` 셀 53

In [ ]:
# ───── 원본: TRPG_Rule · 셀 52 ─────
# Ex43. 방식별 비교 (개발용 48문항 = golden_ko 28 + golden_new 20)
# allq = golden_ko + golden_new
# cache = {g["q"]: (rewrite_query(g["q"]), rewrite_query2c(g["q"])) for g in allq}   # 번역은 한 번씩만
#
# def rrf(q, k=5, c0=60):
#     old, new = cache[q]
#     lists = [search_two([t], 10) for t in (q, old, new)]
#     score, byid = {}, {}
#     for lst in lists:
#         for r, c in enumerate(lst, 1):
#             score[c["id"]] = score.get(c["id"], 0) + 1 / (c0 + r)
#             byid[c["id"]] = c
#     return [byid[i] for i in sorted(score, key=score.get, reverse=True)[:k]]
#
# fns = {
#     "A 한국어 그대로":        lambda q: search_two([q]),
#     "B 예전 번역(HINTS)":     lambda q: search_two([cache[q][0]]),
#     "C 용어 선택지":          lambda q: search_two([cache[q][1]]),
#     "D 선택지 + 한국어 보충": lambda q: search_fallback(q, cache[q][1]),
#     "E 세 검색 RRF":          lambda q: rrf(q),
# }
#
# print(f"{'방식':<22}{'golden_ko(28)':<18}{'golden_new(20)':<18}합계(48)")
# for name, fn in fns.items():
#     a = counts([rank_in(fn(g["q"]), g)[0] for g in golden_ko])
#     b = counts([rank_in(fn(g["q"]), g)[0] for g in golden_new])
#     s = tuple(x + y for x, y in zip(a, b))
#     print(f"{name:<22}{str(a):<18}{str(b):<18}{s}")

# ───── 원본: TRPG_Rule · 셀 53 ─────
# Ex44. 최종 검색 함수(RRF) + 질문 함수 + 영어 회귀 확인
# def search_final(q, k=5, c0=60):
#     texts = [q, rewrite_query(q), rewrite_query2c(q)]    # 한국어 그대로 / 예전 번역 / 용어 선택지 번역
#     lists = [search_two([t], 10) for t in texts]
#     score, byid = {}, {}
#     for lst in lists:
#         for r, c in enumerate(lst, 1):                   # 순위가 높을수록 큰 점수를 더함
#             score[c["id"]] = score.get(c["id"], 0) + 1 / (c0 + r)
#             byid[c["id"]] = c
#     top = sorted(score, key=score.get, reverse=True)[:k]
#     return [byid[i] for i in top], texts
#
# def ask_rules(q, answer=True, k=5):
#     found, texts = search_final(q, k)
#     print("바꾼 것:", texts[1], "|", texts[2])
#     print("검색 결과:", [f"{c['meta']['title']} (p.{c['meta']['page']})" for c in found])
#     if not answer:
#         return
#     context = "\n".join(f"[{n}] ({c['meta']['title']} p.{c['meta']['page']}) {c['text'][:2000]}"
#                         for n, c in enumerate(found, 1))
#     prompt = RAG_TEMPLATE.format(terms=TERMS, context=context, question=q)
#     for t in range(3):
#         a = ask_llm(prompt, temperature=0.1 + 0.3 * t)
#         if not has_chinese(a):
#             break
#     print("\n" + a)
#
# 영어 25문항이 나빠지지 않았는지 확인
# ranks = [rank_in(search_final(g["q"])[0], g)[0] for g in golden]
# print("영어 25문항:", counts(ranks), "| 이전 최고 (22, 24, 24) | 원래 (18, 22, 23)")

### ✅ 채택·반영 · 키워드(BM25) 합치기 비교: 지금(E) → 키워드 추가(F) → 키워드 절반(F2)
- 합계(66): E (43,55,58) / F (42,59,60) / F2 (43,57,61). F2(가중 RRF, 키워드 비중 0.5)를 검색 v1로 채택
- 출처: `02_rules_rag` 셀 35, `02_rules_rag` 셀 36

In [ ]:
# ───── 원본: 02_rules_rag · 셀 35 ─────
# Ex12. 하이브리드 검색 비교: 지금(E) vs 키워드 추가(F)
# trans = {}                                             # 번역은 질문마다 한 번만
# def get_trans(q):
#     if q not in trans:
#         trans[q] = (rewrite_query(q), rewrite_query2c(q))
#     return trans[q]
#
# def fuse(lists, k=5, c0=60):                           # 여러 검색 결과를 순위 점수(RRF)로 합치기
#     score, byid = {}, {}
#     for lst in lists:
#         for r, c in enumerate(lst, 1):
#             score[c["id"]] = score.get(c["id"], 0) + 1 / (c0 + r)
#             byid[c["id"]] = c
#     return [byid[i] for i in sorted(score, key=score.get, reverse=True)[:k]]
#
# def search_E(q):                                       # 지금: 임베딩 3갈래
#     en1, en2 = get_trans(q)
#     return fuse([search_two([t], 10) for t in (q, en1, en2)])
#
# def search_F(q):                                       # 임베딩 3갈래 + 키워드 2갈래 (영어 번역 두 개로)
#     en1, en2 = get_trans(q)
#     dense = [search_two([t], 10) for t in (q, en1, en2)]
#     return fuse(dense + [bm25_search(en1, 10), bm25_search(en2, 10)])
#
# sets = {"영어 golden": golden, "한국어 golden_ko": golden_ko, "한국어 golden_new": golden_new}
# ranks = {"E": {}, "F": {}}
# for sname, items in sets.items():
#     for name, fn in (("E", search_E), ("F", search_F)):
#         ranks[name][sname] = [rank_in(fn(g["q"]), g)[0] for g in tqdm(items, leave=False)]
#     print(f"{sname:<18} E {counts(ranks['E'][sname])}  →  F {counts(ranks['F'][sname])}")
#
# 순위가 바뀐 질문 (좋아졌나 나빠졌나)
# for sname, items in sets.items():
#     for g, e, f in zip(items, ranks["E"][sname], ranks["F"][sname]):
#         if e != f:
#             print(f"[{sname}] E {e} → F {f} | {g['q']}")

# ───── 원본: 02_rules_rag · 셀 36 ─────
# Ex13. 키워드 검색을 절반 비중으로 (F2)
# def fuse_w(lists, weights, k=5, c0=60):               # 검색마다 비중(weight)을 줘서 합치기
#     score, byid = {}, {}
#     for lst, w in zip(lists, weights):
#         for r, c in enumerate(lst, 1):
#             score[c["id"]] = score.get(c["id"], 0) + w / (c0 + r)
#             byid[c["id"]] = c
#     return [byid[i] for i in sorted(score, key=score.get, reverse=True)[:k]]
#
# def search_F2(q):
#     en1, en2 = get_trans(q)
#     lists = [search_two([t], 10) for t in (q, en1, en2)] + [bm25_search(en1, 10), bm25_search(en2, 10)]
#     return fuse_w(lists, [1, 1, 1, 0.5, 0.5])         # 임베딩 3표(1.0) + 키워드 2표(0.5)
#
# for sname, items in sets.items():
#     ranks.setdefault("F2", {})[sname] = [rank_in(search_F2(g["q"]), g)[0] for g in tqdm(items, leave=False)]
#
# print(f"{'':<18}{'E (지금)':<18}{'F (+키워드)':<18}F2 (키워드 절반)")
# tot = {k: [0, 0, 0] for k in ("E", "F", "F2")}
# for sname in sets:
#     row = {k: counts(ranks[k][sname]) for k in tot}
#     for k in tot:
#         tot[k] = [a + b for a, b in zip(tot[k], row[k])]
#     print(f"{sname:<18}{str(row['E']):<18}{str(row['F']):<18}{row['F2']}")
# print(f"{'합계(66)':<18}{str(tuple(tot['E'])):<18}{str(tuple(tot['F'])):<18}{tuple(tot['F2'])}")

### 🔍 진단 · 번역을 14B로 바꿔 같은 표 다시 재기 (이때는 생성 모델은 7B 유지)
- 합계(66): 7B F2 (43,57,61) / 14B E (42,53,53) / 14B F2 (42,55,58) → 14B 번역이 더 좋아지진 않음
- 이후 14B 하나로 통일할 때 다시 쟀고 (43,52,59)로 7B와 비슷했음 (본문 Ex08-c)
- 출처: `02_rules_rag` 셀 37, `02_rules_rag` 셀 38

In [ ]:
# ───── 원본: 02_rules_rag · 셀 37 ─────
# !nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
# !ollama pull qwen2.5:14b

# ───── 원본: 02_rules_rag · 셀 38 ─────
# Ex14. 번역만 14B로 바꿔서 같은 표 다시 재기 (답변 생성 모델은 그대로)
# TRANS_MODEL = "qwen2.5:14b"
# _ask_plain_7b = ask_plain
# def ask_plain(prompt, model=None, temperature=0.0):          # 번역 함수들이 이 함수를 부름
#     return _ask_plain_7b(prompt, model=model or TRANS_MODEL, temperature=temperature)
#
# trans14 = {}                                                  # 14B 번역은 따로 저장 (7B 번역 캐시 trans는 그대로 둠)
# trans_7b, trans = trans, trans14                              # get_trans가 쓰는 캐시를 임시로 교체
#
# ranks14 = {}
# for sname, items in sets.items():
#     ranks14[sname] = {
#         "E":  [rank_in(search_E(g["q"]),  g)[0] for g in tqdm(items, leave=False)],
#         "F2": [rank_in(search_F2(g["q"]), g)[0] for g in tqdm(items, leave=False)],
#     }
#
# tot = {"7B E": [0,0,0], "7B F2": [0,0,0], "14B E": [0,0,0], "14B F2": [0,0,0]}
# print(f"{'':<18}{'7B E':<16}{'7B F2':<16}{'14B E':<16}14B F2")
# for sname in sets:
#     row = {"7B E": counts(ranks["E"][sname]), "7B F2": counts(ranks["F2"][sname]),
#            "14B E": counts(ranks14[sname]["E"]), "14B F2": counts(ranks14[sname]["F2"])}
#     for k in tot: tot[k] = [a + b for a, b in zip(tot[k], row[k])]
#     print(f"{sname:<18}{str(row['7B E']):<16}{str(row['7B F2']):<16}{str(row['14B E']):<16}{row['14B F2']}")
# print(f"{'합계(66)':<18}{str(tuple(tot['7B E'])):<16}{str(tuple(tot['7B F2'])):<16}{str(tuple(tot['14B E'])):<16}{tuple(tot['14B F2'])}")
#
# trans = trans_7b                                              # 원래 캐시로 복구
# ask_plain = _ask_plain_7b                                     # 원래 번역 함수로 복구

### 🔍 진단 · Prone 질문: 원문 그대로 vs 번역 vs 최종 검색 / 용어표(힌트·선택지)를 빼도 되는지 비교
- Prone 질문은 원문·번역①·번역②·최종 모두 순위 None → 알려진 약점
- 합계(66): G0 힌트+선택지 (43,57,61) / G1 힌트만 뺌 (44,56,59) / G2 둘 다 뺌 (34,52,53)
- → 힌트만 빼도 비슷해서 번역 ①은 힌트 없는 일반 번역으로 확정
- 출처: `02_rules_rag2` 셀 20, `02_rules_rag2` 셀 28

In [ ]:
# ───── 원본: 02_rules_rag2 · 셀 20 ─────
# Ex08b. Prone 질문: 원문 그대로 vs 번역 vs 최종 검색
# prone_items = [g for s in (golden, golden_ko, golden_new) for g in s if "Prone" in wants_of(g)]
# for g in prone_items:
#     q = g["q"]
#     found, texts = search_final(q)
#     raw_rank   = rank_in(search_two([q], 10), g)[0]            # 질문 원문만 (번역 없음)
#     en1_rank   = rank_in(search_two([texts[1]], 10), g)[0]     # 번역 ①만
#     en2_rank   = rank_in(search_two([texts[2]], 10), g)[0]     # 번역 ②만
#     fin_rank   = rank_in(found, g)[0]
#     print(f"{q}\n   번역: {texts[1]} | {texts[2]}\n   순위(원문/번역①/번역②/최종): {raw_rank}/{en1_rank}/{en2_rank}/{fin_rank}\n")

# ───── 원본: 02_rules_rag2 · 셀 28 ─────
# Ex08b. 용어표(힌트·선택지)를 빼도 되는지 비교
# REWRITE_PLAIN = """Convert the user's question about D&D 5.2 rules into a short English search query.
# Use the official rule terms that the rulebook would use.
# Output only the query, nothing else.
#
# Question: 숙련 보너스는 레벨마다 얼마야?
# Query: Proficiency Bonus by character level
#
# Question: 레벨업하려면 경험치가 얼마나 필요해?
# Query: Experience Points needed to advance level
#
# Question: 방어도는 어떻게 계산해?
# Query: Armor Class calculation
#
# Question: {question}
# Query:"""
#
# def rewrite_plain(q):                                  # 번역 ③ 용어표 없이 일반 번역
#     return clean_query(ask_plain(REWRITE_PLAIN.format(question=q)))
#
# TR = {"hint": rewrite_query, "plain": rewrite_plain, "closed": rewrite_query2c}
# cache_tr = {}
# def tr(kind, q):                                       # 같은 번역은 한 번만 호출
#     if (kind, q) not in cache_tr:
#         cache_tr[(kind, q)] = TR[kind](q)
#     return cache_tr[(kind, q)]
#
# def make_fn(kinds):                                    # kinds = 쓸 번역 종류들
#     def fn(q):
#         ens = [tr(k, q) for k in kinds]
#         lists = [search_two([t], 10) for t in [q] + ens] + [bm25_search(e, 10) for e in ens]
#         return fuse_w(lists, [1] * (1 + len(ens)) + [0.5] * len(ens))
#     return fn
#
# variants = {
#     "G0 지금(힌트+선택지)":      ["hint", "closed"],
#     "G1 힌트만 뺌(일반+선택지)":  ["plain", "closed"],
#     "G2 둘 다 뺌(일반 하나)":    ["plain"],
# }
# sets = {"영어": golden, "한국어ko": golden_ko, "한국어new": golden_new}
#
# for name, kinds in variants.items():
#     res = {s: evaluate(make_fn(kinds), items) for s, items in sets.items()}
#     total = tuple(sum(v[i] for v in res.values()) for i in range(3))
#     print(f"{name:26s}", res, "| 합계(66):", total)

### ❌ 기각 · 재순위(reranker) 진단: 재순위가 1등을 뺏은 질문 + 최종 세트 실험
- 최종 세트: 기준 (31,46,51) vs R1 재순위 (28,47,50) → 1등이 오히려 줄어 채택 안 함
- 재순위 실험 셀 자체는 본문의 '참고: 재순위' 구간에 주석으로 있어요
- 출처: `02_rules_rag2` 셀 42, `02_rules_rag2` 셀 43

In [ ]:
# ───── 원본: 02_rules_rag2 · 셀 42 ─────
# Ex16. 재순위가 1등을 뺏은 질문 보기 (진단용, 고치지 않음)
# fn0, fn1 = variants["기준(재순위 없음)"], variants["R1 재순위(한국어 질문)"]
# for sname, items in sets.items():
#     for g in items:
#         r0 = rank_in(fn0(g["q"]), g)[0]
#         r1, t1 = rank_in(fn1(g["q"]), g)
#         if r0 == 1 and r1 != 1:
#             print(f"[{sname}] {g['q']}\n   정답: {g['want']} | 재순위 후: {r1} | 상위3: {t1[:3]}")

# ───── 원본: 02_rules_rag2 · 셀 43 ─────
# Ex17. final세트에서 재순위 실험
# rr_path = f"{DATA}/final_result_rerank.json"
# assert not os.path.exists(rr_path), "이미 했어요. 다시 하려면 이유를 기록하고 파일을 지우세요."
#
# res = {}
# for name, fn in {"기준": variants["기준(재순위 없음)"], "R1 재순위": variants["R1 재순위(한국어 질문)"]}.items():
#     rows = []
#     for g in tqdm(golden_final, desc=name):
#         r, titles = rank_in(fn(g["q"]), g)
#         rows.append({"q": g["q"], "rank": r, "top3": titles[:3]})
#     res[name] = {"counts": counts([x["rank"] for x in rows]), "rows":rows}
#     print(name, res[name]["counts"], "/", len(golden_final))
#
# json.dump(res, open(rr_path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)

### 🔍 진단 · 재현성 진단: 같은 프롬프트 4번 + 중간에 다른 프롬프트 끼우기
- 고정한 호출(temperature 0 + seed 42)은 4번 모두 같고, 다른 프롬프트를 끼워도 같음 → 실험용 `gen`의 재현성 확인
- (나중에 챗봇용 `ask_llm`은 seed가 실제로 안 전달되던 문제를 따로 발견해서 고침)
- 출처: `02_rules_rag2` 셀 46, `02_rules_rag2` 셀 47

In [ ]:
# ───── 원본: 02_rules_rag2 · 셀 46 ─────
# Ex19-b. 실험용 생성 호출 : 매번 같은 결과가 나오도록 고정, 재시도 없음
# def gen(prompt, model, system=SYSTEM):
#     res = requests.post(
#         f"{OLLAMA_URL}/api/chat",
#         json={"model": model,
#               "messages": [{"role": "system", "content": system},
#                            {"role": "user", "content": prompt}],
#               "stream": False,
#               "options": {"num_ctx": 8192, "temperature": 0, "seed": 42}},
#         timeout=300,
#     )
#     res.raise_for_status()
#     return res.json()["message"]["content"]
#
# 확인 : 같은 질문을 두 번 돌려 결과가 같은지
# g = dev[0]
# ctx = "\n".join(f"[{n}] ({c['meta']['title']} p.{c['meta']['page']}) {c['text'][:2000]}"
#                 for n, c in enumerate(g["gold"], 1))
# p = RAG_TEMPLATE.format(terms=TERMS, context=ctx, question=g["q"])
# a1, a2 = gen(p, LLM_MODEL), gen(p, LLM_MODEL)
# print(a1)
# print("두 번 결과 동일:", a1 == a2)

# ───── 원본: 02_rules_rag2 · 셀 47 ─────
# Ex19-c. 재현성 진단: 같은 프롬프트 4번 + 중간에 다른 프롬프트 끼우기
# p_other = RAG_TEMPLATE.format(terms=TERMS, context=ctx, question="전혀 다른 질문이에요")
#
# a = [gen(p, LLM_MODEL) for _ in range(4)]
# print("1,2,3,4번 서로 같은지:", [a[0] == x for x in a], "| 2번=3번:", a[1] == a[2], "| 3번=4번:", a[2] == a[3])
#
# gen(p_other, LLM_MODEL)                         # 다른 프롬프트를 사이에 넣어 캐시를 바꿈
# b = gen(p, LLM_MODEL)
# print("끼운 뒤 결과 = 1번:", b == a[0], "| = 4번:", b == a[3])
#
# for i, x in enumerate(a, 1):
#     print(f"--- {i}번\n{x}")

### 🔍 진단 · 저장된 v1 점수(31/46/51)를 재현하는 구성 찾기
- 처음엔 22/60 문항 순위가 달라지고 현재 구성은 (38,49,54)로 나옴
- 원인 추적: 당시 번역과 지금 번역 비교, 옛 노트북 정의로 재현 시도 → 호출 옵션 차이
- 결론: `num_ctx 4096` 설정에서 번역이 60/60 같고 v1 (31,46,51)이 순위 60/60 재현됨 → 본문의 번역 호출이 ctx 4096인 이유
- 개발셋 비교(66): v1 (41,54,57) vs 일반+열린 번역 (37,50,52) → v1 구성 유지
- 출처: `02_rules_rag2` 셀 48, `02_rules_rag2` 셀 49, `02_rules_rag2` 셀 51, `02_rules_rag2` 셀 52, `02_rules_rag2` 셀 53, `02_rules_rag2` 셀 56

In [ ]:
# ───── 원본: 02_rules_rag2 · 셀 48 ─────
# Ex20. search_final을 확정 v1 구성으로 교체 (일반 번역 + 열린 번역)
# def search_final(q, k=5):
#     en1, en2 = rewrite_plain(q), rewrite_open(q)
#     lists = [search_two([t], 10) for t in (q, en1, en2)] + [bm25_search(en1, 10), bm25_search(en2, 10)]
#     return fuse_w(lists, [1, 1, 1, 0.5, 0.5], k), [q, en1, en2]
#
# 검증: 저장된 v1 결과(final_result_v1.json)와 순위가 같은지 비교 (파일은 읽기만 해요)
# golden_final = load_set("golden_final")
# saved = {x["q"]: x["rank"] for x in json.load(open(f"{DATA}/final_result_v1.json", encoding="utf-8"))["rows"]}
# diff = []
# for g in tqdm(golden_final):
#     r, _ = rank_in(search_final(g["q"])[0], g)
#     if r != saved[g["q"]]:
#         diff.append((g["q"], saved[g["q"]], r))
# print("순위가 달라진 문항:", len(diff), "/", len(golden_final))   # 0이면 확정 v1이 그대로 복원된 거예요
# print(diff[:5])

# ───── 원본: 02_rules_rag2 · 셀 49 ─────
# Ex20-b. 어느 구성이 저장된 v1 순위를 재현하는지 찾기
# saved_rows = json.load(open(f"{DATA}/final_result_v1.json", encoding="utf-8"))["rows"]
# saved = {x["q"]: x["rank"] for x in saved_rows}
# print("저장된 v1:", counts([x["rank"] for x in saved_rows]))
#
# 1) 질문마다 재료(번역 2개 + 검색 결과)를 한 번만 만들어 두기
# mat = {}
# for g in tqdm(golden_final):
#     q = g["q"]
#     en1, en2 = rewrite_plain(q), rewrite_open(q)
#     mat[q] = {"en": (en1, en2),
#               "dense": [search_two([t], 10) for t in (q, en1, en2)],     # 원문, 번역①, 번역② 임베딩 검색
#               "bm25":  [bm25_search(en1, 10), bm25_search(en2, 10)]}     # 번역①, ② 키워드 검색
#
# 2) 구성 후보 (가중치 리스트 순서 = dense 3개 → bm25)
# configs = {
#     "현재 (dense3 + bm25 둘, 0.5)": lambda m: fuse_w(m["dense"] + m["bm25"], [1, 1, 1, .5, .5]),
#     "bm25 번역②만":                lambda m: fuse_w(m["dense"] + [m["bm25"][1]], [1, 1, 1, .5]),
#     "bm25 번역①만":                lambda m: fuse_w(m["dense"] + [m["bm25"][0]], [1, 1, 1, .5]),
#     "bm25 없음":                   lambda m: fuse_w(m["dense"], [1, 1, 1]),
#     "원문 dense 빼고 번역만":        lambda m: fuse_w(m["dense"][1:] + m["bm25"], [1, 1, .5, .5]),
#     "bm25 가중 1":                 lambda m: fuse_w(m["dense"] + m["bm25"], [1, 1, 1, 1, 1]),
# }
# for name, f in configs.items():
#     ranks = [rank_in(f(mat[g["q"]]), g)[0] for g in golden_final]
#     same = sum(r == saved[g["q"]] for r, g in zip(ranks, golden_final))
#     print(f"{name:26s} {counts(ranks)} | 저장과 순위 같은 문항 {same}/60")
#
# 3) 번역이 재현되는지: 앞 10문항을 다시 번역해서 비교
# qs = list(mat)[:10]
# same_tr = sum(mat[q]["en"] == (rewrite_plain(q), rewrite_open(q)) for q in qs)
# print("번역 재현:", same_tr, "/ 10")

# ───── 원본: 02_rules_rag2 · 셀 51 ─────
# Ex20-c. 당시 번역(final_answers_v1.json) vs 지금 번역
# old = {a["q"]: a["queries"] for a in json.load(open(f"{DATA}/final_answers_v1.json", encoding="utf-8"))}
# same1 = same2 = both = 0
# for q in old:
#     o1, o2 = old[q]; n1, n2 = mat[q]["en"]
#     same1 += o1 == n1; same2 += o2 == n2; both += (o1, o2) == (n1, n2)
# print(f"번역① 같음 {same1}/60 | 번역② 같음 {same2}/60 | 둘 다 같음 {both}/60")
#
# diff = [q for q in old if tuple(old[q]) != mat[q]["en"]][:5]
# for q in diff:
#     print(q, "\n   당시:", old[q], "\n   지금:", list(mat[q]["en"]))

# ───── 원본: 02_rules_rag2 · 셀 52 ─────
# Ex20-d. 옛 노트북(TRPG_Rule) 정의로 v1 재현
# TERM_MAP_OLD = {   # 옛 용어표 (행동 포함, 매력)
#     "실명": "Blinded", "매료": "Charmed", "귀먹음": "Deafened", "탈진": "Exhaustion",
#     "공포": "Frightened", "붙잡힘": "Grappled", "행동 불능": "Incapacitated", "투명": "Invisible",
#     "마비": "Paralyzed", "석화": "Petrified", "중독": "Poisoned", "넘어짐": "Prone",
#     "포박": "Restrained", "기절": "Stunned", "의식 불명": "Unconscious",
#     "공격": "Attack", "질주": "Dash", "이탈": "Disengage", "회피": "Dodge", "도움": "Help",
#     "숨기": "Hide", "영향": "Influence", "마법": "Magic", "준비": "Ready", "탐색": "Search",
#     "연구": "Study", "사용": "Utilize",
#     "이점": "Advantage", "불리점": "Disadvantage", "능력 판정": "Ability Check",
#     "공격 굴림": "Attack Roll", "내성 굴림": "Saving Throw", "기회 공격": "Opportunity Attack",
#     "반응": "Reaction", "추가 행동": "Bonus Action", "집중": "Concentration",
#     "이니셔티브": "Initiative", "저항": "Resistance", "취약": "Vulnerability",
#     "엄폐": "Cover", "치명타": "Critical Hit", "숙련": "Proficiency", "전문화": "Expertise",
#     "긴 휴식": "Long Rest", "짧은 휴식": "Short Rest",
#     "근력": "Strength", "민첩": "Dexterity", "건강": "Constitution",
#     "지능": "Intelligence", "지혜": "Wisdom", "매력": "Charisma",
# }
# def hints_of(tm):
#     return ", ".join(f"{ko}={en}" for ko, en in tm.items())
#
# REWRITE_HINT = """Convert the user's question about D&D 5.2 rules into a short English search query.
# Use the official rule terms that the rulebook would use. Korean-to-English term hints: {hints}
# Output only the query, nothing else.
#
# Question: 숙련 보너스는 레벨마다 얼마야?
# Query: Proficiency Bonus by character level
#
# Question: 레벨업하려면 경험치가 얼마나 필요해?
# Query: Experience Points needed to advance level
#
# Question: 방어도는 어떻게 계산해?
# Query: Armor Class calculation
#
# Question: {question}
# Query:"""
#
# RULE_TERMS = ["Blinded", "Charmed", "Deafened", "Exhaustion", "Frightened", "Grappled", "Incapacitated",
#               "Invisible", "Paralyzed", "Petrified", "Poisoned", "Prone", "Restrained", "Stunned", "Unconscious",
#               "Attack action", "Dash action", "Disengage action", "Dodge action", "Help action", "Hide action",
#               "Influence action", "Magic action", "Ready action", "Search action", "Study action", "Utilize action"]
#
# REWRITE2 = """Translate the user's D&D question into a short English search query for the D&D rules.
# If the question is about one of these official rule terms, use that term exactly as written.
# Official terms: {terms}
# Output only the query, nothing else.
#
# Question: {q}
# Query:"""
#
# def rewrite_hint(q, tm):                              # 옛 번역① (힌트 방식)
#     out = ask_plain(REWRITE_HINT.format(hints=hints_of(tm), question=q))
#     return out.strip().split("\n")[0].replace("Query:", "").strip()
#
# def rewrite_closed(q):                                # 옛 번역② (용어 선택지 방식)
#     return clean_query(ask_plain(REWRITE2.format(terms=", ".join(RULE_TERMS), q=q)))
#
# 1) 당시 번역과 같은지
# qs = list(old)
# m_closed = sum(rewrite_closed(q) == old[q][1] for q in qs)
# m_hint = {name: sum(rewrite_hint(q, tm) == old[q][0] for q in qs)
#           for name, tm in {"옛 용어표(행동 포함)": TERM_MAP_OLD, "새 용어표(행동 제외)": TERM_MAP}.items()}
# print("번역② 용어 선택지 방식 일치:", m_closed, "/ 60")
# print("번역① 힌트 방식 일치:", m_hint)
#
# 2) 일치가 높은 힌트 용어표로 v1 전체(번역 + 검색 + 합치기)를 재현해서 저장된 순위와 비교
# best = max(m_hint, key=m_hint.get)
# tm_best = TERM_MAP_OLD if best.startswith("옛") else TERM_MAP
# mat2 = {}
# for g in tqdm(golden_final):
#     q = g["q"]; en1, en2 = rewrite_hint(q, tm_best), rewrite_closed(q)
#     mat2[q] = {"dense": [search_two([t], 10) for t in (q, en1, en2)],
#                "bm25":  [bm25_search(en1, 10), bm25_search(en2, 10)]}
# cfg = {"dense3 + bm25(0.5)": lambda m: fuse_w(m["dense"] + m["bm25"], [1, 1, 1, .5, .5]),
#        "dense3만 (옛 파일 방식)": lambda m: fuse_w(m["dense"], [1, 1, 1])}
# for name, f in cfg.items():
#     ranks = [rank_in(f(mat2[g["q"]]), g)[0] for g in golden_final]
#     same = sum(r == saved[g["q"]] for r, g in zip(ranks, golden_final))
#     print(f"{best} | {name}: {counts(ranks)} | 저장과 순위 같은 문항 {same}/60")

# ───── 원본: 02_rules_rag2 · 셀 53 ─────
# Ex20-e. 호출 설정 차이 확인 + 일반 번역 + 용어 선택지로 v1 재현
# def ask_opts(prompt, **opts):
#     o = {"temperature": 0.0, **opts}
#     res_ = requests.post(f"{OLLAMA_URL}/api/chat",
#         json={"model": LLM_MODEL, "messages": [{"role": "user", "content": prompt}],
#               "stream": False, "options": o}, timeout=300)
#     res_.raise_for_status()
#     return res_.json()["message"]["content"]
#
# def tr1(q, **o): return clean_query(ask_opts(REWRITE_TEMPLATE.format(question=q), **o))      # ① 일반 번역
# def tr2(q, **o): return clean_query(ask_opts(REWRITE2.format(terms=", ".join(RULE_TERMS), q=q), **o))   # ② 용어 선택지
#
# qs = list(old)
# opts_list = {"seed42+ctx8192": {"seed": 42, "num_ctx": 8192},
#              "ctx4096":         {"num_ctx": 4096},
#              "ctx2048":         {"num_ctx": 2048},
#              "옵션 없음":        {}}
# score_opt = {}
# for name, o in opts_list.items():
#     m1 = sum(tr1(q, **o) == old[q][0] for q in qs)
#     m2 = sum(tr2(q, **o) == old[q][1] for q in qs)
#     score_opt[name] = m1 + m2
#     print(f"{name:16s} ①{m1}/60 ②{m2}/60")
#
# 가장 많이 일치한 설정으로 v1 전체 재현
# best = max(score_opt, key=score_opt.get); ob = opts_list[best]
# print("선택된 설정:", best)
# mat3 = {}
# for g in tqdm(golden_final):
#     q = g["q"]; en1, en2 = tr1(q, **ob), tr2(q, **ob)
#     mat3[q] = {"en": (en1, en2),
#                "dense": [search_two([t], 10) for t in (q, en1, en2)],
#                "bm25":  [bm25_search(en1, 10), bm25_search(en2, 10)]}
# cfg = {"dense3 + bm25(0.5)": lambda m: fuse_w(m["dense"] + m["bm25"], [1, 1, 1, .5, .5]),
#        "dense3만":           lambda m: fuse_w(m["dense"], [1, 1, 1])}
# for name, f in cfg.items():
#     ranks = [rank_in(f(mat3[g["q"]]), g)[0] for g in golden_final]
#     same = sum(r == saved[g["q"]] for r, g in zip(ranks, golden_final))
#     print(f"{name}: {counts(ranks)} | 저장과 순위 같은 문항 {same}/60")

# ───── 원본: 02_rules_rag2 · 셀 56 ─────
# Ex20-g. 개발셋에서만: v1 vs 일반+열린 번역 (ctx4096 / ctx8192)
# def tro(q, **o): return clean_query(ask_opts(REWRITE_OPEN.format(q=q), **o))   # 열린 번역
#
# def cfg_fn(t1, t2, **o):
#     def fn(q):
#         en1, en2 = t1(q, **o), t2(q, **o)
#         lists = [search_two([t], 10) for t in (q, en1, en2)] + [bm25_search(en1, 10), bm25_search(en2, 10)]
#         return fuse_w(lists, [1, 1, 1, .5, .5])
#     return fn
#
# configs = {"v1 (일반+선택지, ctx4096)": cfg_fn(tr1, tr2, num_ctx=4096),
#            "일반+열린, ctx4096":        cfg_fn(tr1, tro, num_ctx=4096),
#            "일반+열린, ctx8192":        cfg_fn(tr1, tro, num_ctx=8192)}
# sets = {"영어": golden, "한국어ko": golden_ko, "한국어new": golden_new}
# for name, fn in configs.items():
#     r = {s: evaluate(fn, items) for s, items in sets.items()}
#     total = tuple(sum(v[i] for v in r.values()) for i in range(3))
#     print(f"{name:26s} {r} | 합계(66): {total}")

### 🔍 진단 · 7B·14B 설명에서 깨짐/숫자 의심 문항 직접 보기, 문항 목록
- 7B 현행: 깨짐 4/38 (한자·일본어 섞임), Hide 설명에서 'Three-Quarters'가 '3분의 4'로 나옴. 14B(좁힘)에서도 '3분의 2'와 일부 한자 깨짐이 나옴
- → 이 관찰이 `MIXED`/`FRACTION` 검사와 재생성(`bad_output`)으로 이어짐
- 출처: `02_rules_rag2` 셀 58, `02_rules_rag2` 셀 62

In [ ]:
# ───── 원본: 02_rules_rag2 · 셀 58 ─────
# Ex21-c. 0번 문항 정리 + 깨짐/숫자 의심 문항 직접 보기
# def refresh_first(name, key, template):
#     g = dev[0]
#     p = template.format(terms=TERMS, context=build_ctx(g), question=g["q"])
#     gen(p, "qwen2.5:7b")                              # 실제 프롬프트로 워밍업 (결과 버림)
#     a, b = gen(p, "qwen2.5:7b"), gen(p, "qwen2.5:7b")
#     print(name, "워밍업 뒤 두 번 같음:", a == b)
#     res[key][0]["answer"] = b
#     path = f"{DATA}/expl_{name}.json"
#     d = json.load(open(path, encoding="utf-8")); d["rows"] = res[key]
#     json.dump(d, open(path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
#
# refresh_first("A_7B_current", "A_7B_현행", RAG_TEMPLATE)
# refresh_first("B_7B_narrow",  "B_7B_좁힘", NARROW_TEMPLATE)
#
# 깨짐으로 잡힌 글자와 문장
# for k, rows in res.items():
#     print(f"\n######## {k}")
#     for r, g in zip(rows, dev):
#         e = expl_part(r["answer"])
#         bad = BROKEN.findall(e)
#         if bad:
#             print(f"[잡힌 글자 {sorted(set(bad))}] {g['q']}\n   {e[:300]}")
#
# 근거에 없는 숫자가 나온 문항
# for k, rows in res.items():
#     for r, g in zip(rows, dev):
#         e = expl_part(r["answer"]); src = set(numbers(src_text(g)))
#         extra = [n for n in numbers(e) if n not in src]
#         if extra:
#             print(f"\n[{k}] {g['q']} | 근거에 없는 숫자 {extra}\n   설명: {e}\n   근거 속 숫자: {sorted(src)[:15]}")
#
# for k, rows in res.items():
#     print(k); score(rows)

# ───── 원본: 02_rules_rag2 · 셀 62 ─────
# Ex23-a. 문항 목록(번호, 질문, 정답 제목) + 14B 의심 문항 전문
# for i, g in enumerate(dev):
#     print(i, "|", g["q"], "|", g["want"], "|", [c["meta"]["title"] for c in g["gold"]])
#
# for k in ("C_14B_현행", "D_14B_좁힘"):
#     print(f"\n######## {k}")
#     for i, (r, g) in enumerate(zip(res[k], dev)):
#         e = expl_part(r["answer"])
#         extra = [n for n in numbers(e) if n not in src_numbers(g)]
#         if is_broken(e) or extra or LEAK.search(e):
#             print(f"[{i}] 깨짐={sorted(set(BROKEN.findall(e)))} 숫자={extra}\n   {e}")

### 🔍 진단 · 14B 근거 선택 평가(한 번에 하기) + 호출 1번이 느린 이유 구간별 측정
- 한 번에 하기 14B: 영어 20/24, 한국어ko 18/23, 한국어new 8/9 (본문 Ex14-c에서 비교 기준으로 씀)
- 호출 1번: 검색 8.0초 중 모델 올리기(load) 5.4초 → 모델 교체가 느림의 원인, 그래서 검색 결과를 캐시
- 출처: `02_rules_rag2` 셀 64, `02_rules_rag2` 셀 65

In [ ]:
# ───── 원본: 02_rules_rag2 · 셀 64 ─────
# Ex24-a. 번역 캐시(7B로 한 번) → 14B로 근거 선택 평가
# if "_orig_q1" not in globals():                      # 셀을 다시 돌려도 이중으로 감싸지 않게
#     _orig_q1, _orig_q2 = rewrite_query, rewrite_query2c
# _tr_cache = globals().get("_tr_cache", {})
# def rewrite_query(q):
#     if ("1", q) not in _tr_cache: _tr_cache[("1", q)] = _orig_q1(q)
#     return _tr_cache[("1", q)]
# def rewrite_query2c(q):
#     if ("2", q) not in _tr_cache: _tr_cache[("2", q)] = _orig_q2(q)
#     return _tr_cache[("2", q)]
#
# for items in (golden, golden_ko, golden_new):        # 7B로 번역을 미리 만들어 두기
#     for g in tqdm(items, leave=False):
#         search_final(g["q"])
#
# requests.post(f"{OLLAMA_URL}/api/generate", json={"model": "qwen2.5:7b", "keep_alive": 0})   # 7B 내리기
# LLM_MODEL = "qwen2.5:14b"                            # 이후 생성(ask_llm)은 14B
#
# all_rows14 = {}
# for name, items in {"영어": golden, "한국어ko": golden_ko, "한국어new": golden_new}.items():
#     all_rows14[name] = eval_pick(items)              # 셀 39의 eval_pick (14B로 답변 생성)
#     rows = all_rows14[name]
#     H = [x for x in rows if x["hit_rank"]]
#     C = [x for x in H if x["ok"]]
#     print(f"{name}: 검색 적중 {len(H)}/{len(rows)} | 그중 근거 선택 정답 {len(C)}/{len(H)}   (7B: 영어 21/24, ko 15/23, new 8/10)")

# ───── 원본: 02_rules_rag2 · 셀 65 ─────
# Ex24-b. 호출 1번이 왜 느린지 구간별로 재기
# q = golden_new[0]["q"]
# t0 = time.time(); found, texts = search_final(q); t1 = time.time()      # 검색(임베딩 모델 사용)
# ctx = "\n".join(f"[{n}] ({c['meta']['title']} p.{c['meta']['page']}) {c['text'][:2000]}"
#                 for n, c in enumerate(found, 1))
# prompt = RAG_TEMPLATE.format(terms=TERMS, context=ctx, question=q)
# print("검색:", round(t1 - t0, 1), "초 | 프롬프트:", len(prompt), "자")
#
# r = requests.post(f"{OLLAMA_URL}/api/chat",
#     json={"model": LLM_MODEL, "messages": [{"role": "system", "content": SYSTEM},
#                                            {"role": "user", "content": prompt}],
#           "stream": False, "options": {"num_ctx": 8192, "temperature": 0.1}}, timeout=600).json()
# for k in ("load_duration", "prompt_eval_duration", "eval_duration"):
#     print(k, round(r[k] / 1e9, 1), "초")
# print("입력 토큰", r["prompt_eval_count"], "| 생성 토큰", r["eval_count"])
# !ollama ps

### 🧪 단순 테스트 · 14B 받기 + 7B GPU에서 내리기, 최종 세트 풀링 파일 저장
- 단순 작업 코드 (모델 정리, 판정용 파일 만들기)
- 출처: `02_rules_rag2` 셀 60, `02_rules_rag2` 셀 73

In [ ]:
# ───── 원본: 02_rules_rag2 · 셀 60 ─────
# Ex22-a. 14B 받기 + 7B를 GPU에서 내리기
# !ollama pull qwen2.5:14b
# keep_alive 0 = 쓴 직후 모델을 메모리에서 내려요 (수업에 없음. T4 메모리를 비우려는 용도)
# requests.post(f"{OLLAMA_URL}/api/generate", json={"model": "qwen2.5:7b", "keep_alive": 0})
# !ollama ps

# ───── 원본: 02_rules_rag2 · 셀 73 ─────
# Ex18. 풀링 : 기준과 R1이 가져온 상위 5개를 모아 판정용 파일로 저장 (최종 세트)

### ♻️ 더 나은 코드로 대체 · 초기 최종 출력(검증·거절 없음) 및 단순 점검 코드
- Ex15-e 초기 버전: 검사에 걸리면 설명을 숨기고 원문만 보여 줌 → 근거 검증(거절) + 충실성 검사를 넣은 `answer_final`로 대체
- 용어표 후보 뽑기 / 피해 종류 추가 / 서버 재가동 / seed 적용 확인은 이미 용어표 v1·본문에 반영돼 단순 점검 코드만 남김
- 출처: `rules_rag_clean` 셀 32, `rules_rag_clean` 셀 33, `rules_rag_clean` 셀 59, `rules_rag_clean` 셀 60, `rules_rag_clean` 셀 62, `rules_rag_clean` 셀 68

In [ ]:
# ───── 원본: rules_rag_clean · 셀 32 ─────
# Ex09-a2. 용어표에 피해 종류 추가 (규칙상 닫힌 집합 13개. 한국어 표기는 팀에서 확정하세요)
# TERM_MAP.update({"산성": "Acid", "타격": "Bludgeoning", "냉기": "Cold", "화염": "Fire", "역장": "Force",
#                  "번개": "Lightning", "사령": "Necrotic", "관통": "Piercing", "독": "Poison", "정신": "Psychic",
#                  "광휘": "Radiant", "참격": "Slashing", "천둥": "Thunder"})
# TERMS = ", ".join(f"{en}={ko}" for ko, en in TERM_MAP.items())
# TERM_REV = {en.lower(): ko for ko, en in TERM_MAP.items()}

# ───── 원본: rules_rag_clean · 셀 33 ─────
# Ex09-a3. 룰북 용어집 후보 뽑기 (용어표에 없는 것만 보여요)
# gloss = [c for c in chunks if "Rules Glossary" in c["text"].split("\n")[0]]
# print("Rules Glossary 청크:", len(gloss))
#
# have = {en.lower() for en in TERM_MAP.values()}
# cands = sorted({c["meta"]["title"] for c in gloss if c["meta"]["title"].lower() not in have})
# print("용어표에 없는 용어 후보:", len(cands))
# print(cands[:80])

# ───── 원본: rules_rag_clean · 셀 59 ─────
# 올라마 서버 재가동
# os.environ["OLLAMA_KEEP_ALIVE"] = "1h"
# subprocess.Popen("nohup ollama serve > ollama.log 2>&1 &", shell=True)
# wait_server(OLLAMA_URL)
# !ollama list

# ───── 원본: rules_rag_clean · 셀 60 ─────
# import inspect
# print(inspect.signature(ask_llm))                      # seed=42 가 보이면 ask_llm은 적용된 것
# print("seed" in inspect.getsource(rag_answer2))        # True면 rag_answer2도 적용된 것
#
# q = "What roll do you make to hide from enemies?"
# outs = [rag_answer2(q)["explain"] for _ in range(3)]   # 같은 질문 3번
# print("3번 모두 같은지:", [o == outs[0] for o in outs])

# ───── 원본: rules_rag_clean · 셀 62 ─────
# Ex15-e. 최종 출력: 검사에 걸린 설명은 숨기고 원문 + 페이지만 보여줘요 (용어 누락은 참고용이라 판정에 안 써요)
# def answer_final(question):
#     r = rag_answer2(question)
#     chk = check_answer2(r)
#     warn = []
#     if chk["근거에_없는_숫자"]: warn.append("숫자")
#     if chk["분수"]: warn.append("분수")
#     if chk["혼합_철자"]: warn.append("혼합 철자")
#     if is_broken(r["explain"]): warn.append("깨짐")
#     return {"question": question, "title": r["title"], "page": r["page"], "path": r["path"],
#             "orig": r["orig"], "orig_is_model": r["orig_ok"],
#             "explain": None if warn else r["explain"], "body": r["text"].split("\n", 1)[-1][:600], "warn": warn, "chk": chk}
#
# def show_final(question):
#     a = answer_final(question)
#     print(f"Q. {question}")
#     print(f"출처: {a['title']} (p.{a['page']})  [{a['path']}]")
#     print(f"원문{'' if a['orig_is_model'] else ' (근거 앞부분)'}: {a['orig']}")
#     if a["explain"]:
#         print(f"설명: {a['explain']}")
#     else:
#         print(f"설명: (자동 검사에서 불확실해서 숨겼어요: {', '.join(a['warn'])}) 아래 근거 본문을 확인해 주세요")
#         print(f"근거 본문: {a['body']}")
#
# for q in ["What roll do you make to hide from enemies?", "전투 중에 기습당하면 어떻게 돼?", "높은 데서 뛰어내리면 얼마의 피해를 받지?"]:
#     show_final(q); print()

# ───── 원본: rules_rag_clean · 셀 68 ─────
# for q in ["무기 두 개를 양손에 들고 싸울 수 있어?", "How far can a creature with darkvision see?", "전투 중에 기습당하면 어떻게 돼?"]:
#     show_final(q); print()